# SpiderNet tutorial: Pancancer example

This notebook demonstrates the **core SpiderNet workflow** on the **Pancancer dataset** and can be used as a **template for applying SpiderNet to a new dataset**.

## Core pipeline
1. **Edit the dataset setup cell**  
   Specify the **input paths**, **resource files**, and key dataset-specific settings.

2. **Check paths and AnnData fields**  
   Verify that all required files, metadata fields, and **AnnData structure** are correctly configured before running the pipeline.

3. **Preprocess the raw data**  
   Convert the input data into the **SpiderNet input format** needed for downstream model training.

4. **Train the model**  
   Run **SpiderNet training** to learn latent **meta-interactions (MIs)** and communication-related representations.

5. **Infer and export results**  
   Generate and save the main outputs, including **MI factors**, **gene / LR loadings**, and other downstream analysis files.

## Optional analyses
- **MI correlation**
- **LR pathway enrichment**
- **sender--receiver cell-type enrichment**

## Before you start

Under `DATA_ROOT`, keep an `adata/` folder with one `.h5ad` file per sample or batch.

Each `.h5ad` should contain:
- Spatial coordinates: `adata.obsm['spatial']`
- Sample IDs: `adata.obs[SAMPLE_ID_COL]`
- Pre-determined cell type labels: `adata.obs[CELL_TYPE_COL]`
- gene symbols in `adata.var_names` (for example `MYC`, `TP53`)



## 0. Dataset setup (edit this cell first)

Put the dataset-specific settings here. In most cases, this is the main cell you need to edit when switching to a new dataset.


In [1]:
from pathlib import Path
import numpy as np

# Paths
DATA_ROOT = Path("D:/SpiderNet/Data/Pancancer")      # path to raw input data
OUTPUT_ROOT = Path("D:/SpiderNet/Results/Pancancer") # path to save outputs and results
ADATA_FOLDER_NAME = "adata_entire"                   # choose from "adata" or "adata_entire"

# Dataset-specific fields
SPECIES = "human"               # "human" or "mouse"
SAMPLE_ID_COL = "SampleID"      # column in adata.obs
CELL_TYPE_COL = "celltype_final" # column in adata.obs
SPATIAL_KEY = "spatial"         # key in adata.obsm

# Preprocessing parameters
N_HVG = 1000
N_HVG_LR = 2000
NUM_NEIGHBORS = 5

# Model parameters
DIM_ENVIR = 11
N_JOBS = 5
MAX_EPOCH = 20000

VERSION = "V1"

# Reference result directory from the subsampled 40-slice training run.
REFERENCE_RESULTS_DIR = Path(
    "D:/SpiderNet/Results/Pancancer/V1/SpiderNet_Result_dim11"
)

# Trained model checkpoint. Set to a specific file, or leave as None to auto-pick
# the latest model_epoch* checkpoint under REFERENCE_RESULTS_DIR / "Model".
TRAINED_MODEL_PATH = None

# Reuse the exact LR list and gene names saved in the reference results.
USE_REFERENCE_LR_LIST = True
USE_REFERENCE_GENENAMES = True
REFERENCE_LR_LIST_FILENAME = "LR_list.pkl"
REFERENCE_GENENAME_CANDIDATES = ["genenames.pkl", "genenames_train.pkl"]


## 1. Imports and device setup

Load SpiderNet utilities and choose CPU or CUDA.


In [2]:
import gc
import json
import subprocess
import sys

import pandas as pd
import torch

from SpiderNet.utils import *
from SpiderNet.config import *

cuda_available = torch.cuda.is_available()
device = "cuda" if cuda_available else "cpu"
print(f"Using device: {device}")


def release_memory(*var_names, namespace=None, run_gc=True, clear_cuda=True, close_figures=False):
    """
    Delete variables from the provided namespace and optionally trigger
    Python garbage collection / CUDA cache cleanup.
    """
    if namespace is None:
        namespace = globals()

    for name in var_names:
        if name in namespace:
            try:
                del namespace[name]
            except Exception:
                namespace.pop(name, None)

    if close_figures:
        try:
            import matplotlib.pyplot as plt
            plt.close("all")
        except Exception:
            pass

    if run_gc:
        gc.collect()

    if clear_cuda and torch.cuda.is_available():
        torch.cuda.empty_cache()


Using device: cuda


## 2. Build paths from the dataset setup

This cell uses the values from the dataset setup cell above.


In [3]:
paths = PathConfig(
    data_root=DATA_ROOT,
    output_root=OUTPUT_ROOT,
    species=SPECIES,
    version=VERSION,
)

paths


PathConfig(data_root=WindowsPath('D:/SpiderNet/Data/Pancancer'), output_root=WindowsPath('D:/SpiderNet/Results/Pancancer'), species='human', cellchat_db=WindowsPath('D:/SpiderNet/SpiderNet_proj/SpiderNet_Project/SpiderNet/resources/Human_LR_pairs_Cellchatdb.csv'), scseqcomm_db=WindowsPath('D:/SpiderNet/SpiderNet_proj/SpiderNet_Project/SpiderNet/resources/Human_LR_pairs_scSeqComm.csv'), version='V1')

In [4]:
# Copy sample IDs and cell type labels to the expected obs fields if needed,
# and check that the expected fields are present.
import scanpy as sc

adata_dir = Path(DATA_ROOT) / ADATA_FOLDER_NAME
sample_files = sorted(adata_dir.glob("*.h5ad"))

updated_rows = []

for sample_path in sample_files:
    adata_i = sc.read_h5ad(sample_path)

    ok_sample_id_col = SAMPLE_ID_COL in adata_i.obs.columns
    ok_cell_type_col = CELL_TYPE_COL in adata_i.obs.columns

    if ok_sample_id_col:
        adata_i.obs["SAMPLE_ID"] = adata_i.obs[SAMPLE_ID_COL].copy()
    if ok_cell_type_col:
        adata_i.obs["CELL_TYPE"] = adata_i.obs[CELL_TYPE_COL].copy()

    adata_i.write_h5ad(sample_path)

    updated_rows.append({
        "file": sample_path.name,
        f"has obs['{SAMPLE_ID_COL}']": ok_sample_id_col,
        f"has obs['{CELL_TYPE_COL}']": ok_cell_type_col,
        f"created obs['SAMPLE_ID']": ok_sample_id_col,
        f"created obs['CELL_TYPE']": ok_cell_type_col,
    })

display(pd.DataFrame(updated_rows))


C:\Users\junji\miniconda3\envs\SpiderNet_env\Lib\site-packages\louvain\__init__.py:54: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound


,file,has obs['SampleID'],has obs['celltype_final'],created obs['SAMPLE_ID'],created obs['CELL_TYPE']
0,HumanBreastCancerPatient1_subslice_0_annotated...,False,True,False,True
1,HumanBreastCancerPatient1_subslice_10_annotate...,False,True,False,True
2,HumanBreastCancerPatient1_subslice_11_annotate...,False,True,False,True
3,HumanBreastCancerPatient1_subslice_12_annotate...,False,True,False,True
4,HumanBreastCancerPatient1_subslice_13_annotate...,False,True,False,True
...,...,...,...,...,...
155,HumanUterineCancerPatient1_subslice_5_annotate...,False,True,False,True
156,HumanUterineCancerPatient1_subslice_6_annotate...,False,True,False,True
157,HumanUterineCancerPatient1_subslice_7_annotate...,False,True,False,True
158,HumanUterineCancerPatient1_subslice_8_annotate...,False,True,False,True


### Check paths and AnnData fields

Use the next cell to confirm that paths exist and the first `.h5ad` file has the expected fields.


In [5]:
adata_dir = Path(DATA_ROOT) / ADATA_FOLDER_NAME
sample_files = sorted(adata_dir.glob("*.h5ad"))
sample_path = sample_files[0] if sample_files else None

rows = [
    {"item": "DATA_ROOT", "value": str(DATA_ROOT), "ok": Path(DATA_ROOT).exists()},
    {"item": "OUTPUT_ROOT", "value": str(OUTPUT_ROOT), "ok": Path(OUTPUT_ROOT).exists()},
    {"item": "ADATA_FOLDER_NAME", "value": ADATA_FOLDER_NAME, "ok": adata_dir.exists()},
    {"item": "CellChat DB", "value": str(paths.cellchat_db), "ok": Path(paths.cellchat_db).exists()},
    {"item": "scSeqComm DB", "value": str(paths.scseqcomm_db), "ok": Path(paths.scseqcomm_db).exists()},
    {"item": "adata folder", "value": str(adata_dir), "ok": adata_dir.exists()},
    {"item": "number of .h5ad files", "value": len(sample_files), "ok": len(sample_files) > 0},
    {"item": "REFERENCE_RESULTS_DIR", "value": str(REFERENCE_RESULTS_DIR), "ok": REFERENCE_RESULTS_DIR.exists()},
]

if sample_path is not None:
    adata_example = sc.read_h5ad(sample_path, backed="r")
    rows.extend([
        {"item": "example file", "value": sample_path.name, "ok": True},
        {"item": f"obs['{SAMPLE_ID_COL}']", "value": SAMPLE_ID_COL, "ok": SAMPLE_ID_COL in adata_example.obs.columns},
        {"item": f"obs['{CELL_TYPE_COL}']", "value": CELL_TYPE_COL, "ok": CELL_TYPE_COL in adata_example.obs.columns},
        {"item": f"obsm['{SPATIAL_KEY}']", "value": SPATIAL_KEY, "ok": SPATIAL_KEY in adata_example.obsm_keys()},
    ])
    adata_example.file.close()
else:
    rows.append({"item": "example file", "value": "No .h5ad file found", "ok": False})

display(pd.DataFrame(rows))


C:\Users\junji\AppData\Local\Temp\ipykernel_5512\837918054.py:22: FutureWarning: Use obsm (e.g. `k in adata.obsm` or `adata.obsm.keys() | {'u'}`) instead of AnnData.obsm_keys, AnnData.obsm_keys is deprecated and will be removed in the future.
  {"item": f"obsm['{SPATIAL_KEY}']", "value": SPATIAL_KEY, "ok": SPATIAL_KEY in adata_example.obsm_keys()},


,item,value,ok
0,DATA_ROOT,D:\SpiderNet\Data\Pancancer,True
1,OUTPUT_ROOT,D:\SpiderNet\Results\Pancancer,True
2,ADATA_FOLDER_NAME,adata_entire,True
3,CellChat DB,D:\SpiderNet\SpiderNet_proj\SpiderNet_Project\...,True
4,scSeqComm DB,D:\SpiderNet\SpiderNet_proj\SpiderNet_Project\...,True
5,adata folder,D:\SpiderNet\Data\Pancancer\adata_entire,True
6,number of .h5ad files,160,True
7,REFERENCE_RESULTS_DIR,D:\SpiderNet\Results\Pancancer\V1\SpiderNet_Re...,True
8,example file,HumanBreastCancerPatient1_subslice_0_annotated...,True
9,obs['SampleID'],SampleID,False


## 3. Build preprocessing and training configs

These values are taken from the dataset setup cell.


In [6]:
preprocess_cfg = PreprocessConfig(
    n_hvg=N_HVG,
    n_hvg_lr=N_HVG_LR,
    num_neighbors=NUM_NEIGHBORS
)

train_cfg = TrainingConfig(
    dim_envir=DIM_ENVIR,
    n_jobs=N_JOBS,
    max_epoch=MAX_EPOCH,
    version=VERSION
)

run_dirs = paths.ensure_dirs(
    dim_envir=train_cfg.dim_envir
)
print(run_dirs)

with open(paths.output_root / "run_dirs.json", "w", encoding="utf-8") as handle:
    json.dump({k: str(v) for k, v in run_dirs.items()}, handle, indent=2)


{'run_dir': WindowsPath('D:/SpiderNet/Results/Pancancer/V1/SpiderNet_Result_dim11'), 'model_dir': WindowsPath('D:/SpiderNet/Results/Pancancer/V1/SpiderNet_Result_dim11/Model')}


In [7]:
import re

def resolve_reference_genename_path(reference_results_dir, candidates):
    for candidate in candidates:
        candidate_path = reference_results_dir / candidate
        if candidate_path.exists():
            return candidate_path
    raise FileNotFoundError(
        "Cannot find any reference gene-name file under "
        f"{reference_results_dir}. Checked: {candidates}"
    )


def resolve_model_checkpoint(reference_results_dir, explicit_model_path=None):
    if explicit_model_path is not None:
        explicit_model_path = Path(explicit_model_path)
        if not explicit_model_path.exists():
            raise FileNotFoundError(f"Model checkpoint does not exist: {explicit_model_path}")
        return explicit_model_path

    search_dirs = []
    if (reference_results_dir / "Model").exists():
        search_dirs.append(reference_results_dir / "Model")
    search_dirs.append(reference_results_dir)

    candidates = []
    for search_dir in search_dirs:
        for pattern in ["model_epoch*", "*checkpoint*", "*.pt", "*.pth", "*.pkl"]:
            candidates.extend(search_dir.glob(pattern))

    candidates = [p for p in candidates if p.is_file()]
    if len(candidates) == 0:
        raise FileNotFoundError(
            "Cannot find a trained model checkpoint under "
            f"{reference_results_dir} or {reference_results_dir / 'Model'}"
        )

    def extract_epoch(path):
        match = re.search(r"epoch(\d+)", path.stem)
        return int(match.group(1)) if match else -1

    candidates = sorted(
        candidates,
        key=lambda p: (extract_epoch(p), p.stat().st_mtime)
    )
    return candidates[-1]


reference_lr_list_path = (
    REFERENCE_RESULTS_DIR / REFERENCE_LR_LIST_FILENAME
    if USE_REFERENCE_LR_LIST else None
)
if reference_lr_list_path is not None and not reference_lr_list_path.exists():
    raise FileNotFoundError(f"Reference LR list not found: {reference_lr_list_path}")

reference_genename_path = (
    resolve_reference_genename_path(REFERENCE_RESULTS_DIR, REFERENCE_GENENAME_CANDIDATES)
    if USE_REFERENCE_GENENAMES else None
)

reference_model_path = resolve_model_checkpoint(
    REFERENCE_RESULTS_DIR,
    TRAINED_MODEL_PATH
)

display(pd.DataFrame([
    {
        "item": "reference_lr_list_path",
        "value": str(reference_lr_list_path) if reference_lr_list_path is not None else "None",
        "ok": True if reference_lr_list_path is None else reference_lr_list_path.exists(),
    },
    {
        "item": "reference_genename_path",
        "value": str(reference_genename_path) if reference_genename_path is not None else "None",
        "ok": True if reference_genename_path is None else reference_genename_path.exists(),
    },
    {
        "item": "reference_model_path",
        "value": str(reference_model_path),
        "ok": reference_model_path.exists(),
    },
]))


,item,value,ok
0,reference_lr_list_path,D:\SpiderNet\Results\Pancancer\V1\SpiderNet_Re...,True
1,reference_genename_path,D:\SpiderNet\Results\Pancancer\V1\SpiderNet_Re...,True
2,reference_model_path,D:\SpiderNet\Results\Pancancer\V1\SpiderNet_Re...,True


## 4. Preprocess raw data and load processed data

Convert raw input data into SpiderNet-ready objects and save them to the run directory.


In [ ]:
subprocess.run(
    [
        sys.executable,
        "-m",
        "SpiderNet.dataloading_Pancancer",
        str(paths.data_root),
        str(preprocess_cfg.n_hvg),
        str(preprocess_cfg.n_hvg_lr),
        str(paths.cellchat_db),
        str(paths.scseqcomm_db),
        str(preprocess_cfg.num_neighbors),
        str(paths.output_root),
        str(str(run_dirs["run_dir"]) + "/"),
        str(reference_lr_list_path) if reference_lr_list_path is not None else "None",
        str(reference_genename_path) if reference_genename_path is not None else "None",
        ADATA_FOLDER_NAME,
    ],
    check=True,
)


Reloads processed objects for training and downstream analysis.

Check that the number of batches, LR pairs, and training genes is reasonable.


In [8]:
from SpiderNet.io import load_processed_data
processed = load_processed_data(run_dirs["run_dir"])

print("Number of batches:", len(processed.spidernet_data))
print("Number of LR pairs:", len(processed.lr_list))
print("Number of training genes:", processed.genenames_train.shape[0])


Number of batches: 160
Number of LR pairs: 106
Number of training genes: 500


Check whether the processed data are non-empty and internally consistent before training.


In [9]:
summary_rows = []
for i in range(len(processed.adata_list)):
    adata_i = processed.adata_list[i]
    edge_index_max = torch.max(processed.spidernet_data[i]["edge_index"]).cpu().item()
    summary_rows.append(
        {
            "batch_index": i,
            "n_cells": adata_i.n_obs,
            "n_genes": adata_i.n_vars,
            "edge_index_max": edge_index_max,
            "edge_index_matches_n_cells": adata_i.n_obs == (edge_index_max + 1),
        }
    )

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

issues = []
if len(processed.spidernet_data) == 0:
    issues.append("No processed batches were loaded.")
if len(processed.lr_list) == 0:
    issues.append("No ligand-receptor pairs were retained.")
if processed.genenames_train.shape[0] == 0:
    issues.append("No training genes were retained.")
if not summary_df["edge_index_matches_n_cells"].all():
    issues.append("At least one batch has an edge index / cell-count mismatch.")

if issues:
    print("Sanity check flagged the following issues:")
    for issue in issues:
        print("-", issue)
else:
    print("Sanity checks passed. The processed data appear internally consistent.")


,batch_index,n_cells,n_genes,edge_index_max,edge_index_matches_n_cells
0,0,47423,500,47422,True
1,1,40606,500,40605,True
2,2,30419,500,30418,True
3,3,43955,500,43954,True
4,4,30654,500,30653,True
...,...,...,...,...,...
155,155,47838,500,47837,True
156,156,28462,500,28461,True
157,157,47280,500,47279,True
158,158,31677,500,31676,True


Sanity checks passed. The processed data appear internally consistent.


## 5. Build and train the SpiderNet model

Initialize the model using the processed data and training settings.


In [10]:
from SpiderNet.api import build_model
model = build_model(
    processed=processed,
    train_cfg=train_cfg,
    device=device,
)

print(model)


Number of GPUs available: 1
GPU 0: NVIDIA GeForce RTX 4070
SpiderNet_model(
  (dropout_fun): Dropout(p=0.1, inplace=False)
  (Relu): ReLU()
  (Sigmoid): Sigmoid()
  (enc_factor_envir_pre_receiver): Sequential(
    (0): Linear(in_features=500, out_features=128, bias=True)
    (1): Tanh()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): Tanh()
  )
  (enc_factor_envir_pre_sender): Sequential(
    (0): Linear(in_features=500, out_features=128, bias=True)
    (1): Tanh()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): Tanh()
  )
  (enc_factor_envir): Sequential(
    (0): Linear(in_features=128, out_features=128, bias=True)
    (1): Tanh()
    (2): Linear(in_features=128, out_features=11, bias=True)
  )
)


Runs model training and saves checkpoints.

In [11]:
checkpoint = torch.load(reference_model_path, map_location=device)

if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
    state_dict = checkpoint["model_state_dict"]
elif isinstance(checkpoint, dict) and "state_dict" in checkpoint:
    state_dict = checkpoint["state_dict"]
elif isinstance(checkpoint, dict):
    state_dict = checkpoint
else:
    raise ValueError(
        "Unsupported checkpoint format. Expected a state-dict-like object, "
        f"but got: {type(checkpoint)}"
    )

load_result = model.load_state_dict(state_dict, strict=True)
model = model.to(device)
model.eval()

print(f"Loaded pretrained model from: {reference_model_path}")
print(load_result)


release_memory(
    "checkpoint",
    "state_dict",
    "load_result",
    namespace=globals(),
    run_gc=True,
    clear_cuda=False
)


Loaded pretrained model from: D:\SpiderNet\Results\Pancancer\V1\SpiderNet_Result_dim11\Model\model_epoch19999.pth
<All keys matched successfully>


### Extract the inferred meta-interactions and loadings

This step generates the main outputs:
- MI activities
- LR loadings
- sender loadings
- receiver loadings


In [12]:
from SpiderNet.api import infer_meta_interactions, normalize_outputs
results = infer_meta_interactions(model=model, processed=processed)
results = normalize_outputs(results)

print("Factor_envir shape:", results["factor_envir"].shape)
print("LR loading shape:", results["loading_lr"].shape)
print("Receiver loading shape:", results["loading_receiver"].shape)
print("Sender loading shape:", results["loading_sender"].shape)


Factor_envir shape: (23538755, 11)
LR loading shape: (11, 106)
Receiver loading shape: (11, 500)
Sender loading shape: (11, 500)


Saves the main outputs and config files.

In [14]:
from SpiderNet.api import export_results_V0
export_results_V0(
    results=results,
    processed=processed,
    output_dir=run_dirs["run_dir"],
)

with open(run_dirs["model_dir"] / "SpiderNet_model_config.json", "w", encoding="utf-8") as handle:
    json.dump(train_cfg.to_dict(), handle, indent=2)

with open(run_dirs["model_dir"] / "SpiderNet_preprocess_config.json", "w", encoding="utf-8") as handle:
    json.dump(preprocess_cfg.to_dict(), handle, indent=2)

with open(run_dirs["run_dir"] / "Pancancer_inference_source_config.json", "w", encoding="utf-8") as handle:
    json.dump(
        {
            "adata_folder_name": ADATA_FOLDER_NAME,
            "reference_results_dir": str(REFERENCE_RESULTS_DIR),
            "reference_lr_list_path": str(reference_lr_list_path) if reference_lr_list_path is not None else None,
            "reference_genename_path": str(reference_genename_path) if reference_genename_path is not None else None,
            "reference_model_path": str(reference_model_path),
        },
        handle,
        indent=2,
    )

print(f"Results saved to: {run_dirs['run_dir']}")


# The downstream analysis cells read results from disk, so the in-memory
# result dictionary and model object can be released here.
release_memory(
    "results",
    "model",
    namespace=globals(),
    run_gc=True,
    clear_cuda=True
)


Results saved to: D:\SpiderNet\Results\Pancancer\V1\SpiderNet_Result_dim11


## Key output files

Most important outputs:
- `Factor_envir_use.npy`
- `loading_LR_use.npy`
- `loading_sender_use.npy`
- `loading_receiver_use.npy`

Check these before running optional analyses.


## Optional downstream analyses

The remaining sections help interpret results, but are not required to run SpiderNet on a new dataset.


## Optional 0. Get the bulk level gene expression data & mean MI strength of the tissue for benchmarking the MI deconvolution results

In [16]:
##load data
Factor_envir_use = np.load(run_dirs['run_dir'] / "Factor_envir_use.npy")
batch_cell = pd.read_pickle(run_dirs['run_dir'] / "batch_cell.pkl")

In [22]:
batch_cell_unique = np.unique(batch_cell)
batch_cell_unique.shape

(160,)

In [23]:
Factor_envir_use.shape

(23538755, 11)

In [26]:
##Get the mean MI strength
MI_mean_df = pd.DataFrame(
    0.0,
    columns=[f"MI_{i+1}" for i in range(Factor_envir_use.shape[1])],
    index=batch_cell_unique
)
for batch_cell_cur in batch_cell_unique:
    mask_cur = batch_cell == batch_cell_cur
    mask_cur_index = np.where(mask_cur)[0]
    MI_mean_df.loc[batch_cell_cur, :] = np.mean(Factor_envir_use[mask_cur_index, :], axis=0)

In [29]:
subslice_id = MI_mean_df.index.to_series().str.extract(r'_subslice(\d+)$')[0].astype(int)

MI_mean_df["Split"] = np.where(subslice_id.isin([0, 1, 2, 3, 4]), "Train", "Test")

In [54]:
##Save the MI_mean_df
MI_mean_df.to_csv(run_dirs['run_dir'] / "MI_mean_df.csv")
# MI_mean_df

In [59]:
run_dirs['run_dir'] / "MI_mean_df.csv"

WindowsPath('D:/SpiderNet/Results/Pancancer/V1/SpiderNet_Result_dim11/MI_mean_df.csv')

In [31]:
##Get the bulk level gene expression data
adata_list = pd.read_pickle(run_dirs['run_dir'] / "adata_list.pkl")

In [39]:
for i, adata_i in enumerate(adata_list):
    print(f"Batch {i}: {adata_i.n_obs} cells, {adata_i.n_vars} genes")
    np.mean(adata_i.X.toarray(),axis = 0)

(47423, 500)

In [52]:
geneexp_mean_df = pd.DataFrame(
    0.0,
    columns=adata_list[0].var_names,
    index=batch_cell_unique
)
for batch_index in range(len(adata_list)):
    adata_i = adata_list[batch_index]
    SampleID_cur = adata_i.obs['SampleID'].unique()[0]
    geneexp_mean_df.loc[SampleID_cur, :] = np.mean(adata_i.X.toarray(), axis=0)

In [58]:
##Save the geneexp_mean_df
geneexp_mean_df.to_csv(run_dirs['run_dir'] / "geneexp_mean_df.csv")
geneexp_mean_df

,ACKR3,ACTA2,ADAMTS4,AKT1,AKT2,AKT3,AMOTL2,ANGPT1,ANGPT2,APC,...,WNT3A,WNT5A,WWTR1,XBP1,XCL1,XCR1,YAP1,ZAP70,ZBED2,ZEB1
HumanBreastCancerPatient1_subslice0,0.810905,1.878289,0.488832,2.905039,0.684198,2.761244,1.025149,0.192653,0.220541,1.218002,...,0.341160,0.753166,2.016724,2.443122,0.092801,0.118455,2.604563,0.290228,0.141519,0.561102
HumanBreastCancerPatient1_subslice1,0.854898,1.705814,0.521367,3.089016,0.703833,2.852798,1.098469,0.206580,0.242133,1.307389,...,0.350619,0.772167,2.016103,2.657731,0.101363,0.134895,2.566498,0.322539,0.155790,0.528247
HumanBreastCancerPatient1_subslice10,0.897858,1.982014,0.478422,2.918154,0.712714,2.882178,1.085105,0.216260,0.246612,1.283747,...,0.345184,0.836102,2.098250,2.523108,0.102777,0.128724,2.616055,0.280774,0.146951,0.630236
HumanBreastCancerPatient1_subslice11,0.828871,2.191729,0.619582,2.966501,0.677676,2.754545,1.022047,0.213098,0.258586,1.199412,...,0.365985,0.812246,2.048869,2.424372,0.103550,0.130356,2.647788,0.247569,0.135824,0.692456
HumanBreastCancerPatient1_subslice12,0.837770,2.065384,0.549668,2.939436,0.659759,2.718430,1.045879,0.231859,0.255643,1.235298,...,0.358923,0.749964,1.999740,2.491050,0.104875,0.140119,2.534075,0.301200,0.140023,0.634231
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
HumanUterineCancerPatient1_subslice5,0.567651,1.097275,0.272642,4.012243,0.631600,0.488946,1.269436,0.285957,0.213930,1.385256,...,0.540278,0.836058,2.084712,4.436536,0.089294,0.290377,4.205473,0.382944,0.220691,0.544298
HumanUterineCancerPatient1_subslice6,0.572314,0.972950,0.262527,4.131377,0.633104,0.463415,1.330949,0.305616,0.192954,1.382526,...,0.492298,0.708137,2.188898,4.651507,0.095117,0.249305,4.235403,0.346667,0.203480,0.603882
HumanUterineCancerPatient1_subslice7,0.530864,1.144982,0.277774,3.981164,0.614451,0.423554,1.294364,0.241587,0.203819,1.306596,...,0.445454,0.796141,2.262811,4.640970,0.092338,0.259570,4.033961,0.393261,0.233149,0.559857
HumanUterineCancerPatient1_subslice8,0.513018,1.109849,0.242434,3.849471,0.609736,0.500813,1.183977,0.278885,0.159753,1.311261,...,0.511709,0.756854,1.953396,4.313020,0.080838,0.328944,4.115288,0.367134,0.206867,0.491186


In [56]:
run_dirs['run_dir'] / "geneexp_mean_df.csv"

WindowsPath('D:/SpiderNet/Results/Pancancer/V1/SpiderNet_Result_dim11/geneexp_mean_df.csv')

## Optional 1. MI correlation analysis

Explores relationships among inferred meta-interactions.


In [ ]:
from SpiderNet.analysis import MI_correlation

mi_results = MI_correlation(run_dirs['run_dir'], run_dirs['run_dir'] / "Factor_envir_use.npy", show=True)


release_memory('mi_results', namespace=globals(), run_gc=True, clear_cuda=False)


## Optional 2. LR loading-based pathway enrichment

Interprets each MI using ligand--receptor pathway content.


In [ ]:
from SpiderNet.analysis import LRLoading_enrichment
LRLoading_enrichment(
    loading_LR_use_path=run_dirs['run_dir'] / "loading_LR_use.npy",
    lr_list_path=run_dirs['run_dir'] / "LR_list.pkl",
    lr_list_cellchatdb_path=run_dirs['run_dir'] / "LR_list_cellchatdb.pkl",
    lr_meta_cellchatdb_path=run_dirs['run_dir'] / "LR_meta_cellchatdb.pkl",
    Factor_envir_use_path=run_dirs['run_dir'] / "Factor_envir_use.npy",
    file_savepath_main=run_dirs['run_dir'],
    show=True,
    min_lr_pairs_per_pathway = 2
)


## Optional 3. Sender-cell-type--receiver-cell-type interaction analysis

Summarizes MI enrichment across sender--receiver cell-type pairs.


In [ ]:
from SpiderNet.analysis import MI_Celltypepair_enrichment
MI_Celltypepair_enrichment(
    SpiderNet_data_pyg_list_path=run_dirs['run_dir'] / "SpiderNet_data_pyg_list.pkl",
    Factor_envir_use_path=run_dirs['run_dir'] / "Factor_envir_use.npy",
    file_savepath_main=run_dirs['run_dir'],
    metadata_sample_path="None",
    LR_loading_pathway_path=run_dirs['run_dir'] / "LR_loading_pathway.csv",
    dim_envir=train_cfg.dim_envir,
    MIlevel_agg_threshold=0.6,
    batch_cell_unique_path=run_dirs['run_dir'] / "batch_cell_unique.pkl",
    adata_list_path=run_dirs['run_dir'] / "adata_list.pkl",
    adata_copy_path=run_dirs['run_dir'] / "adata_all.h5ad",
    show=True
)


## Get the mean MI activity log fold change between tumor-invloving and non-tumor-involving interactions

In [ ]:
##load data
Factor_envir_use = np.load(run_dirs['run_dir'] / "Factor_envir_use.npy")
batch_cell = pd.read_pickle(run_dirs['run_dir'] / "batch_cell.pkl")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.patches as mpatches
from scipy.stats import wilcoxon


# ============================================================
# Step 1. Collect edge-level metadata from processed batches
# ============================================================
def collect_edge_metadata(processed):
    """
    Extract, for every edge across all batches:
    1) the slice/sample ID of the sender cell,
    2) the sender cell type,
    3) the receiver cell type.
    """
    edge_sample_list = []
    sender_celltype_list = []
    receiver_celltype_list = []

    for batch_idx in range(len(processed.spidernet_data)):
        data_cur = processed.spidernet_data[batch_idx]
        edge_index_cur = data_cur["edge_index"].cpu().numpy()

        sample_cur = np.asarray(data_cur["sample"]).astype(str)
        celltype_cur = np.asarray(data_cur["cell_class"]).astype(str)

        edge_sample_list.append(sample_cur[edge_index_cur[:, 0]])
        sender_celltype_list.append(celltype_cur[edge_index_cur[:, 0]])
        receiver_celltype_list.append(celltype_cur[edge_index_cur[:, 1]])

    edge_sample = np.hstack(edge_sample_list)
    sender_celltype = np.hstack(sender_celltype_list)
    receiver_celltype = np.hstack(receiver_celltype_list)

    return edge_sample, sender_celltype, receiver_celltype


# ============================================================
# Step 2. Summarize MI activity at the slice level
# ============================================================
def summarize_mi_by_slice(factor_matrix, edge_sample, edge_has_cancer_cell, use_tumor_edges=True):
    """
    Compute slice-level MI summaries using the median MI value across edges.

    Parameters
    ----------
    factor_matrix : np.ndarray
        Edge-by-MI matrix.
    edge_sample : np.ndarray
        Slice/sample ID for each edge.
    edge_has_cancer_cell : np.ndarray of bool
        Whether each edge involves at least one cancer cell.
    use_tumor_edges : bool
        If True, summarize edges that involve cancer cells.
        If False, summarize edges that do not involve cancer cells.
    """
    slice_ids = np.sort(np.unique(edge_sample))
    mi_summary_list = []

    base_mask = edge_has_cancer_cell if use_tumor_edges else ~edge_has_cancer_cell

    for slice_id in slice_ids:
        mask_cur = (edge_sample == slice_id) & base_mask

        if np.any(mask_cur):
            mi_summary = np.median(factor_matrix[mask_cur, :], axis=0)
        else:
            mi_summary = np.full(factor_matrix.shape[1], np.nan)

        mi_summary_list.append(mi_summary)

    mi_summary_df = pd.DataFrame(
        np.vstack(mi_summary_list),
        index=slice_ids,
        columns=[f"MI_{i + 1}" for i in range(factor_matrix.shape[1])]
    )

    return mi_summary_df


# ============================================================
# Step 3. Aggregate slice-level MI summaries to the sample level
# ============================================================
def aggregate_slice_to_sample(mi_slice_df, slice_cell_count):
    """
    Aggregate slice-level MI summaries into sample-level summaries using
    cell-count-weighted averaging across slices from the same sample.
    """
    slice_ids = mi_slice_df.index.astype(str).tolist()
    sample_ids = [slice_id.split("_")[0] for slice_id in slice_ids]
    unique_samples = np.sort(np.unique(sample_ids))

    mi_sample_df = pd.DataFrame(
        np.nan,
        index=unique_samples,
        columns=mi_slice_df.columns,
        dtype=float
    )

    for sample_id in unique_samples:
        slice_ids_cur = [sid for sid in slice_ids if sid.split("_")[0] == sample_id]
        weights_cur = slice_cell_count.loc[slice_ids_cur].values.astype(float)
        values_cur = mi_slice_df.loc[slice_ids_cur, :].values.astype(float)

        # Remove slices with all-NaN MI summaries
        valid_rows = ~np.isnan(values_cur).all(axis=1)
        values_cur = values_cur[valid_rows]
        weights_cur = weights_cur[valid_rows]

        if values_cur.shape[0] == 0:
            continue

        weighted_mean = (
            np.sum(values_cur * weights_cur[:, np.newaxis], axis=0) /
            (np.sum(weights_cur) + 1e-10)
        )
        mi_sample_df.loc[sample_id, :] = weighted_mean

    return mi_sample_df


# ============================================================
# Step 4. Compute tumor and non-tumor slice-level MI summaries
# ============================================================
Factor_envir_use_show = Factor_envir_use

edge_sample, sender_celltype, receiver_celltype = collect_edge_metadata(processed)

# Mark edges that involve at least one cancer cell
edge_has_cancer_cell = np.logical_or(
    np.char.find(sender_celltype, "-cancercell") >= 0,
    np.char.find(receiver_celltype, "-cancercell") >= 0
)

MI_mean_pd_tumor = summarize_mi_by_slice(
    factor_matrix=Factor_envir_use_show,
    edge_sample=edge_sample,
    edge_has_cancer_cell=edge_has_cancer_cell,
    use_tumor_edges=True
)

MI_mean_pd_nontumor = summarize_mi_by_slice(
    factor_matrix=Factor_envir_use_show,
    edge_sample=edge_sample,
    edge_has_cancer_cell=edge_has_cancer_cell,
    use_tumor_edges=False
)

print("Tumor-edge slice-level MI summary:")
display(MI_mean_pd_tumor)

print("Non-tumor-edge slice-level MI summary:")
display(MI_mean_pd_nontumor)


# ============================================================
# Step 5. Get the number of cells in each slice
# ============================================================
# This assumes processed.adata_list is batch-aligned with processed.spidernet_data.
slice_ids_processed = [
    str(np.asarray(processed.spidernet_data[i]["sample"])[0])
    for i in range(len(processed.spidernet_data))
]
numcell_list = [processed.adata_list[i].n_obs for i in range(len(processed.adata_list))]

slice_cell_count = pd.Series(numcell_list, index=slice_ids_processed, dtype=float)

print("Number of cells in each slice:")
display(slice_cell_count)


# ============================================================
# Step 6. Aggregate tumor and non-tumor MI summaries to sample level
# ============================================================
MI_mean_pd_tumor_sample = aggregate_slice_to_sample(
    mi_slice_df=MI_mean_pd_tumor,
    slice_cell_count=slice_cell_count
)

MI_mean_pd_nontumor_sample = aggregate_slice_to_sample(
    mi_slice_df=MI_mean_pd_nontumor,
    slice_cell_count=slice_cell_count
)

print("Tumor-edge sample-level MI summary:")
display(MI_mean_pd_tumor_sample)

print("Non-tumor-edge sample-level MI summary:")
display(MI_mean_pd_nontumor_sample)


# ============================================================
# Step 7. Compute log2(tumor / non-tumor) at the sample level
# ============================================================
MI_mean_pd_tumor_vs_nontumor = (
    MI_mean_pd_tumor_sample + 1e-6
) / (
    MI_mean_pd_nontumor_sample + 1e-6
)

MI_mean_pd_tumor_vs_nontumor_log2 = np.log2(MI_mean_pd_tumor_vs_nontumor)

# Order MIs by the mean log2 ratio across samples
colmean = MI_mean_pd_tumor_vs_nontumor_log2.mean(axis=0)
MI_mean_pd_tumor_vs_nontumor_log2 = MI_mean_pd_tumor_vs_nontumor_log2.loc[
    :, colmean.sort_values(ascending=False).index
]

print("Sample-level log2(tumor / non-tumor):")
display(MI_mean_pd_tumor_vs_nontumor_log2)

MI_mean_pd_tumor_sample_colmax = MI_mean_pd_tumor_sample.max(axis=0)
print("Maximum tumor-edge MI value across samples:")
display(MI_mean_pd_tumor_sample_colmax)


# ============================================================
# Step 8. Plot the maximum tumor-versus-non-tumor difference for each MI
# ============================================================
# Keep MIs with relatively large absolute tumor-edge activity for highlighting
MI_mean_pd_tumor_sample_colmax_large = MI_mean_pd_tumor_sample_colmax[
    MI_mean_pd_tumor_sample_colmax > 0.1
].index.tolist()

print("MIs with tumor-edge sample-level max > 0.1:")
print(MI_mean_pd_tumor_sample_colmax_large)

df = MI_mean_pd_tumor_vs_nontumor_log2.copy()


# Release edge-level helper arrays after slice- and sample-level summaries
# are created. Keep the summary tables because they are reused below.
release_memory(
    "Factor_envir_use_show",
    "edge_sample",
    "sender_celltype",
    "receiver_celltype",
    "edge_has_cancer_cell",
    "slice_ids_processed",
    "numcell_list",
    "slice_cell_count",
    "colmean",
    namespace=globals(),
    run_gc=True,
    clear_cuda=False
)


In [ ]:
# Nature-style plotting settings
mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "axes.linewidth": 0.8,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
    "xtick.major.size": 4,
    "ytick.major.size": 4,
})
sns.set_theme(style="white", context="paper")

# For each MI, use the maximum log2(tumor / non-tumor) value across samples
max_s = df.max(axis=0)

plot_df = max_s.reset_index()
plot_df.columns = ["MI", "max_value"]
plot_df["MI"] = plot_df["MI"].astype(str)

# Order MIs by descending maximum effect size
plot_df = plot_df.sort_values("max_value", ascending=False).reset_index(drop=True)
mi_order = plot_df["MI"].tolist()
df = df.loc[:, mi_order]

print("MI plotting order:")
print(mi_order)

mi_large_set = set([str(x) for x in MI_mean_pd_tumor_sample_colmax_large])

# Optional one-sided Wilcoxon signed-rank test: H1 median > 0
pvals = {}
for mi in mi_order:
    x = df[mi].astype(float).dropna().values

    if x.size == 0 or np.allclose(x, 0):
        pvals[mi] = 1.0
        continue

    try:
        _, p = wilcoxon(
            x,
            zero_method="wilcox",
            alternative="greater",
            mode="auto"
        )
        pvals[mi] = float(p)
    except ValueError:
        pvals[mi] = 1.0

pval_s = pd.Series(pvals)

# Highlight MIs that satisfy both:
# 1) max log2(tumor / non-tumor) > 1
# 2) tumor-edge sample-level max > 0.1
colors = [
    "red" if (row.max_value > 1 and row.MI in mi_large_set) else "lightgrey"
    for _, row in plot_df.iterrows()
]

plt.close("all")
fig, ax = plt.subplots(figsize=(7.2, 5.2))

ax.barh(plot_df["MI"], plot_df["max_value"], color=colors)
ax.invert_yaxis()
ax.axvline(0, linestyle="--", linewidth=1)

ax.set_xlabel("Max log2(tumor / non-tumor) across samples")
ax.set_ylabel("")

red_patch = mpatches.Patch(
    color="red",
    label="max log2 ratio > 1 and tumor-edge max > 0.1"
)
grey_patch = mpatches.Patch(
    color="lightgrey",
    label="other MIs"
)
ax.legend(handles=[red_patch, grey_patch], frameon=False, loc="lower right")

sns.despine(ax=ax)
plt.tight_layout()
plt.savefig(
    str(run_dirs["run_dir"]) + "/MI_tumor_vs_nontumor_max_barplot.pdf",
    bbox_inches="tight"
)
plt.show()
plt.close()

print("One-sided Wilcoxon signed-rank p-values (H1: median > 0):")
display(pval_s.loc[mi_order])

release_memory(
    "df",
    "max_s",
    "plot_df",
    "mi_large_set",
    "pvals",
    "colors",
    "red_patch",
    "grey_patch",
    "fig",
    "ax",
    namespace=globals(),
    run_gc=True,
    clear_cuda=False,
    close_figures=True
)


## Show the mean MI activity for both tumor and non-tumor edges across samples

In [ ]:
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.patches as patches


# ============================================================
# Step 1. Get the number of cells in each slice
# ============================================================
# This assumes that processed.adata_list is aligned with the slice-level MI summary table.
numcell_list = [processed.adata_list[i].n_obs for i in range(len(processed.adata_list))]
print("Number of cells in each slice:")
print(numcell_list)


# ============================================================
# Step 2. Aggregate slice-level MI summaries to the sample level
# ============================================================
def aggregate_slice_mi_to_sample(mi_slice_df, numcell_list):
    """
    Aggregate slice-level MI summaries into sample-level summaries
    using cell-count-weighted averaging.

    Parameters
    ----------
    mi_slice_df : pd.DataFrame
        Slice-level MI summary table.
        Rows are slices and columns are MI features.
    numcell_list : list
        Number of cells in each slice, aligned with mi_slice_df rows.

    Returns
    -------
    pd.DataFrame
        Sample-level MI summary table.
    """
    slice_names = mi_slice_df.index.tolist()
    sample_names = [slice_name.split("_")[0] for slice_name in slice_names]
    unique_samples = np.unique(sample_names)

    mi_sample_df = pd.DataFrame(
        0.0,
        index=unique_samples,
        columns=mi_slice_df.columns
    )

    for sample_name_cur in unique_samples:
        slice_index_cur = [
            i for i in range(len(slice_names))
            if sample_names[i] == sample_name_cur
        ]
        numcell_slice_cur = [numcell_list[i] for i in slice_index_cur]

        weighted_mean = (
            np.sum(
                mi_slice_df.iloc[slice_index_cur, :].values *
                np.array(numcell_slice_cur)[:, np.newaxis],
                axis=0
            ) / (np.sum(numcell_slice_cur) + 1e-10)
        )

        mi_sample_df.loc[sample_name_cur, :] = weighted_mean

    return mi_sample_df


MI_mean_pd_tumor_sample = aggregate_slice_mi_to_sample(
    mi_slice_df=MI_mean_pd_tumor,
    numcell_list=numcell_list
)

MI_mean_pd_nontumor_sample = aggregate_slice_mi_to_sample(
    mi_slice_df=MI_mean_pd_nontumor,
    numcell_list=numcell_list
)

print("Tumor sample-level MI summary:")
display(MI_mean_pd_tumor_sample)

print("Non-tumor sample-level MI summary:")
display(MI_mean_pd_nontumor_sample)

In [ ]:
# ============================================================
# Step 3. Define a reusable heatmap plotting function
# ============================================================
def format_sample_labels(sample_ids):
    """
    Simplify sample labels for plotting.
    """
    sample_ids_new = []
    for sample_id in sample_ids:
        sample_id_new = str(sample_id)

        if "Patient" in sample_id_new:
            sample_id_new = sample_id_new.split("Patient")[0].rstrip("_- ")

        sample_id_new = sample_id_new.replace("Human", "").replace("Cancer", "").strip()
        sample_ids_new.append(sample_id_new)

    return sample_ids_new


def plot_mi_sample_heatmap(
    mi_sample_df,
    mi_order,
    xlabel,
    out_prefix,
    file_savepath_main,
    vmax=0.5,
    highlight_threshold=0.2,
    cmap="inferno"
):
    """
    Plot a sample-level MI heatmap and highlight cells above a threshold.
    """
    mi_sample_df_use = mi_sample_df.copy()

    # Keep only MI columns that appear in the requested order
    mi_order_use = [mi for mi in mi_order if mi in mi_sample_df_use.columns]
    mi_sample_df_use = mi_sample_df_use.loc[:, mi_order_use]

    # Save the reordered matrix
    mi_sample_df_use.to_csv(
        file_savepath_main + "/" + f"{out_prefix}_matrix.csv",
        index=True
    )

    # Prepare plotting labels
    ytick_labels = [
        str(mi).replace("MI_", "MI-")
        for mi in mi_sample_df_use.T.index
    ]
    xtick_labels = format_sample_labels(mi_sample_df_use.T.columns)

    data_plot = mi_sample_df_use.T  # rows = MI, columns = samples

    fig, ax = plt.subplots(figsize=(5, 10))

    hm = sns.heatmap(
        data_plot,
        cmap=cmap,
        vmin=0,
        vmax=vmax,
        ax=ax,
        linewidths=0.30,
        linecolor="#E3E5E6",
        cbar=True,
        rasterized=True,
        cbar_kws=dict(shrink=0.85, aspect=25, pad=0.03)
    )

    # Highlight cells with values above the threshold
    arr = np.asarray(data_plot.values, dtype=float)
    n_rows, n_cols = arr.shape

    for i in range(n_rows):
        for j in range(n_cols):
            if np.isfinite(arr[i, j]) and (arr[i, j] > highlight_threshold):
                rect = patches.Rectangle(
                    (j, i), 1, 1,
                    fill=False,
                    edgecolor="#00F7FF",
                    linewidth=2.0
                )
                ax.add_patch(rect)

    # Apply axis labels and tick labels
    ax.set_yticklabels(ytick_labels, fontsize=12, rotation=0)
    ax.set_xticklabels(xtick_labels, fontsize=11, rotation=60, ha="left")

    ax.xaxis.tick_top()
    ax.xaxis.set_label_position("top")
    ax.tick_params(axis="x", top=True, bottom=False, labeltop=True, labelbottom=False, pad=2)
    ax.tick_params(axis="y", left=True, right=False, pad=2)

    ax.set_xlabel(xlabel, fontsize=14, labelpad=10)
    ax.set_ylabel("Meta-interaction IDs", fontsize=14, labelpad=10)

    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)

    cbar = hm.collections[0].colorbar
    cbar.ax.tick_params(labelsize=10, width=0.8, length=3)

    fig.tight_layout()

    fig.savefig(
        file_savepath_main + "/" + f"{out_prefix}.pdf",
        format="pdf",
        bbox_inches="tight",
        transparent=True
    )
    fig.savefig(
        file_savepath_main + "/" + f"{out_prefix}.png",
        format="png",
        bbox_inches="tight",
        dpi=300,
        transparent=True
    )

    plt.show()
    plt.close(fig)

    # Release local plotting objects before the next heatmap call.
    del arr, data_plot, hm, fig, ax, mi_sample_df_use, ytick_labels, xtick_labels, mi_order_use
    gc.collect()


# ============================================================
# Step 4. Set plotting style
# ============================================================
mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "axes.linewidth": 0.8,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
    "xtick.major.size": 4,
    "ytick.major.size": 4,
})
sns.set_theme(style="white", context="paper")

# from matplotlib.colors import LinearSegmentedColormap
# cmap = LinearSegmentedColormap.from_list(
#     "custom_coolwarm_graycenter",
#     ["#4575b4", "#f0f0f0", "#d73027"],
#     N=256
# )
import matplotlib.pyplot as plt

cmap = plt.get_cmap("YlOrRd")

# ============================================================
# Step 5. Plot tumor sample-level MI heatmap
# ============================================================
plot_mi_sample_heatmap(
    mi_sample_df=MI_mean_pd_tumor_sample,
    mi_order=mi_order,
    xlabel="Cancer sample ID",
    out_prefix="MI_heatmap_tumor_sampleagg",
    file_savepath_main=str(run_dirs['run_dir']),
    vmax=0.3,
    highlight_threshold=0.1,
    cmap=cmap
)

In [ ]:
# ============================================================
# Step 6. Plot non-tumor sample-level MI heatmap
# ============================================================
plot_mi_sample_heatmap(
    mi_sample_df=MI_mean_pd_nontumor_sample,
    mi_order=mi_order,
    xlabel="Non-tumor sample ID",
    out_prefix="MI_heatmap_nontumor_sampleagg",
    file_savepath_main=str(run_dirs['run_dir']),
    vmax=0.3,
    highlight_threshold=0.1,
    cmap=cmap
)

## Re-show the LR pathway enrichment results

In [ ]:
mi_order

In [ ]:
##Load LR_loading_pathway
# LR_loading_pathway.to_csv(os.path.join(file_savepath_main, "LR_loading_pathway.csv"))
LR_loading_pathway = pd.read_csv(run_dirs['run_dir'] / "LR_loading_pathway.csv",index_col=0)
MI_orderbyage = mi_order
##Change the "MI_" to "MI-" in the index of LR_loading_pathway
MI_orderbyage = [x.replace("MI_", "MI-") for x in MI_orderbyage]
LR_loading_pathway = LR_loading_pathway.loc[MI_orderbyage, :]
LR_loading_pathway = LR_loading_pathway.iloc[:,
                     np.argsort(np.array(np.max(LR_loading_pathway, axis=0)))[::-1]
                     ]
# Refined pathway ordering
argmax_1 = np.argmax(np.array(LR_loading_pathway), axis=0)
max_1 = np.max(np.array(LR_loading_pathway), axis=0)
order_index_LRpathway = []
for argmax_1_cur in np.sort(np.unique(argmax_1)):
    index_cur = np.where(argmax_1 == argmax_1_cur)[0]
    index_cur = index_cur[np.argsort(max_1[index_cur])[::-1]]
    index_cur = index_cur.tolist()
    order_index_LRpathway.extend(index_cur)

LR_loading_pathway = LR_loading_pathway.iloc[:, order_index_LRpathway]


In [ ]:
# Visualization
from matplotlib.colors import TwoSlopeNorm, LinearSegmentedColormap
import os
plt.close()
fig, ax = plt.subplots(figsize=(14, 8))
# cmap = LinearSegmentedColormap.from_list(
#     "white_red", ["white", "#FFDFEF", "#EABDE6", "#D69ADE", "#AA60C8"], N=256
# )
cmap = LinearSegmentedColormap.from_list(
    "white_red", ["#FCF5F0", "#F9B2BC", "#F6689F", "#C31988", "#510269"], N=256
)

data = LR_loading_pathway.values
vmin = data.min()
# vmax = min(0.4, np.max(data) * 0.7)
vmax = np.max(data) * 0.6
norm = TwoSlopeNorm(vmin=vmin, vcenter=(vmin + vmax) / 2, vmax=vmax)

mesh = ax.pcolormesh(
    np.arange(data.shape[1] + 1),
    np.arange(data.shape[0] + 1),
    data,
    cmap=cmap,
    norm=norm,
    edgecolors="#B6B9BA",
    linewidth=1.0
)

ax.set_xticks(np.arange(data.shape[1]) + 0.5)
# ax.set_xticklabels(LR_loading_pathway.columns, rotation=60, fontsize=22)
ax.set_xticklabels(
    LR_loading_pathway.columns,
    rotation=60,
    fontsize=22,
    ha="left",
    rotation_mode="anchor"
)
ax.set_yticks(np.arange(data.shape[0]) + 0.5)
ax.set_yticklabels(LR_loading_pathway.index, fontsize=22)
ax.xaxis.set_ticks_position("top")
ax.xaxis.set_label_position("top")
ax.invert_yaxis()

plt.colorbar(mesh, ax=ax)
plt.tight_layout()

save_png = os.path.join(run_dirs['run_dir'], "LR_loading_pathway.png")
plt.savefig(save_png, format="png", bbox_inches="tight", dpi=300)

save_pdf = os.path.join(run_dirs['run_dir'], "LR_loading_pathway.pdf")
plt.savefig(save_pdf, format="pdf", bbox_inches="tight", dpi=300)

plt.show()
plt.close()


release_memory(
    "data",
    "vmin",
    "vmax",
    "norm",
    "mesh",
    "fig",
    "ax",
    namespace=globals(),
    run_gc=True,
    clear_cuda=False,
    close_figures=True
)


In [ ]:
# Release large intermediate objects before the in-situ plotting section.
# Keep `processed`, `run_dirs`, and `mi_order` because they are still needed.
release_memory(
    "Factor_envir_use",
    "batch_cell",
    "MI_mean_pd_tumor",
    "MI_mean_pd_nontumor",
    "MI_mean_pd_tumor_vs_nontumor",
    "MI_mean_pd_tumor_vs_nontumor_log2",
    "MI_mean_pd_tumor_sample",
    "MI_mean_pd_nontumor_sample",
    "MI_mean_pd_tumor_sample_colmax",
    "MI_mean_pd_tumor_sample_colmax_large",
    "pval_s",
    "LR_loading_pathway",
    namespace=globals(),
    run_gc=True,
    clear_cuda=True,
    close_figures=True
)


## in-situ plot of meta-interaction dimension

In [ ]:
##Show the in-situ meta-interaction plot
save_path_insituMI = str(run_dirs['run_dir']) + "/In_situ_meta_interaction/"
if not os.path.exists(save_path_insituMI):
    os.makedirs(save_path_insituMI)

In [ ]:
##MI of interest
MI_OI = "MI-2"

In [ ]:
MI_index = int(MI_OI.replace("MI-", "")) - 1
##
gray_other_cells = True 
##
# vis_mode = 2
vis_mode = 1

In [ ]:
# ============================================================
# In-situ plot of one meta-interaction dimension
# - Plot only one MI at a time
# - Read MI strengths from a memory-mapped .npy file
# - Release per-sample temporary objects immediately after saving
# - Use more conservative default DPI to reduce memory pressure
# - Print progress only every 10% of slices
# ============================================================

Sender_celltype_list_choose = [
    "Breast-cancercell",
    "Colon-cancercell",
    "Liver-cancercell",
    "Lung-cancercell",
    "Melanoma-cancercell",
    "Ovarian-cancercell",
    "Prostate-cancercell",
    "Uterine-cancercell",
]

Receiver_celltype_list_choose = [
    "Endothelial",
    "Fibroblast",
]

import gc
import os
import math
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.patches import FancyArrowPatch

batch_cell_unique = np.asarray(pd.read_pickle(run_dirs["run_dir"] / "batch_cell_unique.pkl"))
factor_envir_path = run_dirs["run_dir"] / "Factor_envir_use.npy"
Factor_envir_use_mmap = np.load(factor_envir_path, mmap_mode="r")

# Pre-compute edge offsets so each sample can read only its own MI slice.
edge_counts = np.asarray(
    [processed.spidernet_data[i]["edge_index"].shape[0] for i in range(len(processed.spidernet_data))],
    dtype=np.int64
)
edge_offsets = np.concatenate(([0], np.cumsum(edge_counts)))

edge_sample_random_state = 2026

if vis_mode == 1:
    base_size = 18
    ratio_size = 1
    arrow_lw_base = 1.2
    arrow_ms = 10
    figure_size = (24, 20)
    dpi_png = 250
    dpi_pdf = 300
elif vis_mode == 2:
    base_size = 10
    ratio_size = 1
    arrow_lw_base = 0.8
    arrow_ms = 8
    figure_size = (24, 20)
    dpi_png = 300
    dpi_pdf = 300
else:
    base_size = 14
    ratio_size = 1
    arrow_lw_base = 1.0
    arrow_ms = 9
    figure_size = (24, 20)
    dpi_png = 250
    dpi_pdf = 300

dot_base = base_size * (2.0 / 3.0)
arrow_ms_eff = arrow_ms * 0.5

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "axes.linewidth": 0.8,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
    "xtick.major.size": 4,
    "ytick.major.size": 4,
})

cell_types = [
    "B cell", "Breast-cancercell", "CD4_T", "CD8_T/NK", "Colon-cancercell", "DC",
    "Endothelial", "Epithelial", "Fibroblast", "Liver-cancercell", "Lung-cancercell",
    "Macrophage", "Mast cell", "Melanoma-cancercell", "Ovarian-cancercell",
    "Prostate-cancercell", "Treg", "Uterine-cancercell", "low_exp",
]
color_set = [
    "#8894F1", "#7F3480", "#FF259E", "#FF52DB", "#F7B2EF", "#2C5E1A",
    "#029991", "#E384FF", "#944E13", "#FF8B6C", "#B37166", "#74650F",
    "#7228B8", "#009D4E", "#DD0303", "#028DCE", "#DE0878", "#FAB12F", "#BBC58F",
]
celltype_to_color = dict(zip(cell_types, color_set))
default_gray = "#BFBFBF"
cmap_edge = LinearSegmentedColormap.from_list("white_to_red", ["white", "red"], N=256)

os.makedirs(save_path_insituMI, exist_ok=True)

# Print progress only at 10%, 20%, ..., 100%
n_samples_total = len(batch_cell_unique)
progress_points = sorted(set([
    max(1, math.ceil(n_samples_total * frac / 10))
    for frac in range(1, 11)
]))

for sample_index_cur, sample_cur in enumerate(batch_cell_unique):
    sample_num = sample_index_cur + 1
    should_report = sample_num in progress_points

    try:
        adata_cursample = processed.adata_list[sample_index_cur]
        data_pyg_cursample = processed.spidernet_data[sample_index_cur]

        spatial_xy = np.asarray(adata_cursample.obsm["spatial"], dtype=np.float32)
        edge_index = data_pyg_cursample["edge_index"].cpu().numpy().astype(np.int32, copy=False)

        if edge_index.ndim != 2 or edge_index.shape[1] != 2:
            raise ValueError(f"edge_index expected shape (E, 2), got {edge_index.shape}")

        start_idx = edge_offsets[sample_index_cur]
        end_idx = edge_offsets[sample_index_cur + 1]
        factor_edge = np.asarray(Factor_envir_use_mmap[start_idx:end_idx, MI_index], dtype=np.float32)

        if factor_edge.shape[0] != edge_index.shape[0]:
            raise ValueError(
                f"Mismatch between edge count and MI edge weights for sample {sample_cur}: "
                f"{edge_index.shape[0]} edges versus {factor_edge.shape[0]} MI values."
            )

        cellclass = np.asarray(data_pyg_cursample["cell_class"]).astype(str)

        src = edge_index[:, 0]
        dst = edge_index[:, 1]
        sender_ct = cellclass[src]
        receiver_ct = cellclass[dst]

        factor_thr = 0.3
        mask_sr = np.isin(sender_ct, Sender_celltype_list_choose) & np.isin(receiver_ct, Receiver_celltype_list_choose)
        mask_thr = factor_edge > factor_thr
        passed_edge_idx = np.flatnonzero(mask_sr & mask_thr)

        n_pass = passed_edge_idx.size
        if n_pass > 2000:
            rng = np.random.default_rng(edge_sample_random_state + sample_index_cur)
            passed_edge_idx = np.sort(rng.choice(passed_edge_idx, size=2000, replace=False))

        if should_report:
            progress_pct = int(round(sample_num / n_samples_total * 100))
            print(f"Progress: {progress_pct}% ({sample_num}/{n_samples_total})")
            # print(f"  passed edges before sampling: {n_pass}")
            # print(f"  edges drawn (after sampling): {passed_edge_idx.size}")

        if passed_edge_idx.size > 0:
            passed_vals = factor_edge[passed_edge_idx]
            if passed_vals.size >= 20:
                vmin = float(np.percentile(passed_vals, 5))
                vmax = float(np.percentile(passed_vals, 95))
            else:
                vmin = float(np.min(passed_vals))
                vmax = float(np.max(passed_vals))
        else:
            passed_vals = factor_edge
            if passed_vals.size == 0:
                vmin, vmax = 0.0, 1.0
            else:
                vmin = float(np.min(passed_vals))
                vmax = float(np.max(passed_vals))

        if np.isclose(vmin, vmax):
            vmax = vmin + 1e-8

        norm_edge = Normalize(vmin=vmin, vmax=vmax, clip=True)

        plt.close("all")
        fig, ax = plt.subplots(figsize=figure_size, facecolor="white")
        ax.set_facecolor("white")

        node_in_edge_mask = np.zeros(len(cellclass), dtype=bool)
        if passed_edge_idx.size > 0:
            involved_nodes = np.unique(np.concatenate([src[passed_edge_idx], dst[passed_edge_idx]]))
            node_in_edge_mask[involved_nodes] = True
        else:
            involved_nodes = np.array([], dtype=np.int32)

        sizes = np.full(len(cellclass), dot_base, dtype=np.float32)
        sizes[node_in_edge_mask] = dot_base * ratio_size

        non_involved_mask = ~node_in_edge_mask
        if np.any(non_involved_mask):
            ax.scatter(
                spatial_xy[non_involved_mask, 0],
                spatial_xy[non_involved_mask, 1],
                c=default_gray,
                s=sizes[non_involved_mask],
                zorder=2,
                rasterized=True,
                alpha=1.0,
                edgecolors="none",
                linewidths=0,
            )

        for ct in cell_types:
            mask_ct = node_in_edge_mask & (cellclass == ct)
            if not np.any(mask_ct):
                continue
            ax.scatter(
                spatial_xy[mask_ct, 0],
                spatial_xy[mask_ct, 1],
                c=celltype_to_color.get(ct, default_gray),
                s=sizes[mask_ct],
                zorder=3,
                rasterized=True,
                alpha=1.0,
                edgecolors="none",
                linewidths=0,
            )

        arrowstyle = "-|>"
        edge_zorder = 10

        for ei in passed_edge_idx:
            s = int(src[ei])
            t = int(dst[ei])

            x0, y0 = spatial_xy[s, 0], spatial_xy[s, 1]
            x1, y1 = spatial_xy[t, 0], spatial_xy[t, 1]

            w01 = float(norm_edge(float(factor_edge[ei])))
            edge_color = cmap_edge(w01)

            arr = FancyArrowPatch(
                (x0, y0),
                (x1, y1),
                arrowstyle=arrowstyle,
                mutation_scale=arrow_ms_eff,
                linewidth=arrow_lw_base * 0.8,
                color=edge_color,
                alpha=0.35 + 0.55 * w01,
                shrinkA=0,
                shrinkB=0,
                capstyle="round",
                joinstyle="round",
                zorder=edge_zorder,
            )
            ax.add_patch(arr)

        ax.set_xticks([])
        ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(False)

        out_png = os.path.join(
            save_path_insituMI,
            f"Insitu_sample{sample_cur}_MI{MI_index + 1}_celltype_Mode{vis_mode}.png"
        )
        out_pdf = os.path.join(
            save_path_insituMI,
            f"Insitu_sample{sample_cur}_MI{MI_index + 1}_celltype_Mode{vis_mode}.pdf"
        )

        fig.savefig(out_png, dpi=dpi_png, bbox_inches="tight", facecolor="white")
        fig.savefig(out_pdf, dpi=dpi_pdf, bbox_inches="tight", facecolor="white")

    finally:
        plt.close("all")
        for _name in [
            "adata_cursample",
            "data_pyg_cursample",
            "spatial_xy",
            "edge_index",
            "factor_edge",
            "cellclass",
            "src",
            "dst",
            "sender_ct",
            "receiver_ct",
            "mask_sr",
            "mask_thr",
            "passed_edge_idx",
            "passed_vals",
            "norm_edge",
            "node_in_edge_mask",
            "involved_nodes",
            "sizes",
            "non_involved_mask",
            "fig",
            "ax",
            "arr",
        ]:
            if _name in globals():
                del globals()[_name]

        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

release_memory(
    "Factor_envir_use_mmap",
    "edge_counts",
    "edge_offsets",
    "batch_cell_unique",
    namespace=globals(),
    run_gc=True,
    clear_cuda=True,
    close_figures=True
)

## Output the result for circle plot of specific MI dimension

In [ ]:
##Load the adata_all
adata_all = sc.read_h5ad(run_dirs['run_dir'] / "adata_all.h5ad")
celltype_all = np.unique(adata_all.obs['celltype_final'].values)
cellclass_unique = np.unique(celltype_all)
target_cellclass_pair_df_all = pd.DataFrame({"Sender": np.repeat(cellclass_unique.tolist(), len(cellclass_unique)).tolist(),
                                         "Receiver": np.tile(cellclass_unique.tolist(), len(cellclass_unique)).tolist()})
# target_cellclass_pair_df_all = target_cellclass_pair_df_all[target_cellclass_pair_df_all['Sender'] != target_cellclass_pair_df_all['Receiver']]
# print(target_cellclass_pair_df_all)
# print(cellclass_unique)

sender_cellclass = []
receiver_cellclass = []
for batch_index in range(len(processed.spidernet_data)):
    edge_index_cur = processed.spidernet_data[batch_index]['edge_index'].to("cpu").numpy()
    cellclass_cursample = np.array(processed.adata_list[batch_index].obs['celltype_final'])
    sender_cellclass_cur = cellclass_cursample[edge_index_cur[:,0]]
    receiver_cellclass_cur = cellclass_cursample[edge_index_cur[:,1]]
    sender_cellclass.extend(sender_cellclass_cur.tolist())
    receiver_cellclass.extend(receiver_cellclass_cur.tolist())
sender_cellclass = np.array(sender_cellclass)
receiver_cellclass = np.array(receiver_cellclass)
# print(sender_cellclass.shape)
# print(receiver_cellclass.shape)
del adata_all, celltype_all, cellclass_unique

In [ ]:
import gc
import numpy as np
import pandas as pd
from scipy import sparse

# --------------------------------------------------
# Fast mean aggregation by (sender, receiver) pair
# using only a random 20% subset of edges
# --------------------------------------------------

sample_ratio = 0.20
random_seed = 2026

# --------------------------------------------------
# Step 1. Read MI matrix with memory mapping
# --------------------------------------------------
factor_path = run_dirs["run_dir"] / "Factor_envir_use.npy"
Factor_envir_use_mmap = np.load(factor_path, mmap_mode="r")
E, M = Factor_envir_use_mmap.shape

# --------------------------------------------------
# Step 2. Randomly sample 20% of edges
# --------------------------------------------------
n_sample = max(1, int(E * sample_ratio))
rng = np.random.default_rng(random_seed)
edge_idx_sample = np.sort(rng.choice(E, size=n_sample, replace=False))

print(f"Original number of edges: {E}")
print(f"Sampled number of edges: {n_sample}")

# Read only the sampled rows into memory
Factor_envir_use = np.asarray(Factor_envir_use_mmap[edge_idx_sample, :], dtype=np.float32)

# sender_cellclass and receiver_cellclass should already exist in memory
sender_arr = np.asarray(sender_cellclass, dtype=object)[edge_idx_sample]
receiver_arr = np.asarray(receiver_cellclass, dtype=object)[edge_idx_sample]

# The full mmap object is no longer needed
del Factor_envir_use_mmap
gc.collect()

# --------------------------------------------------
# Step 3. Define the target sender-receiver pair order
# --------------------------------------------------
pair_list = (
    target_cellclass_pair_df_all["Sender"].astype(str).to_numpy()
    + "_to_"
    + target_cellclass_pair_df_all["Receiver"].astype(str).to_numpy()
)
P = len(pair_list)

# --------------------------------------------------
# Step 4. Build edge-level pair labels
# --------------------------------------------------
sender_str = sender_arr.astype(str)
receiver_str = receiver_arr.astype(str)
edge_pairs = np.char.add(np.char.add(sender_str, "_to_"), receiver_str)

# These object arrays are no longer needed
del sender_arr, receiver_arr, sender_str, receiver_str
gc.collect()

# --------------------------------------------------
# Step 5. Map sampled edges to target pair indices
# --------------------------------------------------
pair_index = pd.Index(pair_list)
codes = pair_index.get_indexer(edge_pairs).astype(np.int32)
mask = codes >= 0

# edge_pairs is no longer needed
del edge_pairs, pair_index
gc.collect()

codes_use = codes[mask]
Factor_use = Factor_envir_use[mask, :]

# These are no longer needed
del codes, mask, Factor_envir_use
gc.collect()

# --------------------------------------------------
# Step 6. Aggregate MI values by pair
# --------------------------------------------------
rows = np.arange(codes_use.size, dtype=np.int64)
cols = codes_use.astype(np.int64)
data = np.ones(codes_use.size, dtype=np.float32)

G = sparse.csr_matrix((data, (rows, cols)), shape=(codes_use.size, P))

# These temporary arrays are no longer needed
del rows, cols, data, codes_use
gc.collect()

sums = G.T.dot(Factor_use)                  # shape = (P, M)
counts = np.asarray(G.sum(axis=0)).ravel() # shape = (P,)

# G and Factor_use are no longer needed after aggregation
del G, Factor_use
gc.collect()

means = np.full_like(sums, np.nan, dtype=np.float32)
np.divide(sums, counts[:, None], out=means, where=(counts[:, None] > 0))

# sums and counts are no longer needed
del sums, counts
gc.collect()

# --------------------------------------------------
# Step 7. Build output matrix
# rows = MI, columns = sender-receiver pairs
# --------------------------------------------------
mean_MI_cellclasspair_all = pd.DataFrame(
    means.T,
    index=[f"MI-{i+1}" for i in range(M)],
    columns=pair_list
)

# means is no longer needed after the DataFrame is created
del means
gc.collect()

# --------------------------------------------------
# Step 8. Export MI-2 links
# --------------------------------------------------
df_plot_all = pd.DataFrame({
    "MI": np.repeat(mean_MI_cellclasspair_all.index, mean_MI_cellclasspair_all.shape[1]),
    "Pair": np.tile(mean_MI_cellclasspair_all.columns, mean_MI_cellclasspair_all.shape[0]),
    "Mean": mean_MI_cellclasspair_all.values.flatten()
})

df_MI2_all = df_plot_all[df_plot_all["MI"] == "MI-2"].copy()
df_MI2_all["Receiver"] = df_MI2_all["Pair"].str.split("_to_").str[1]
df_MI2_all["Sender"] = df_MI2_all["Pair"].str.split("_to_").str[0]

csv_path = str(run_dirs["run_dir"]) + "/MI2_all_links.csv"
df_MI2_all.to_csv(csv_path, index=False)

print(f"Saved: {csv_path}")

# Optional cleanup for large temporary tables
del df_plot_all
gc.collect()

## Optional GO enrichment analyses of the top regulators and targets

This section performs GO enrichment on the top regulators and target genes identified from the loading analysis. 

In [ ]:
import gseapy as gp

In [ ]:
MI_OI = "MI2"

In [ ]:
## Identify candidate regulator and target genes for knockout
loading_receiver_use = np.load(run_dirs['run_dir'] / "loading_receiver_use.npy")
loading_sender_use = np.load(run_dirs['run_dir'] / "loading_sender_use.npy")
loading_receiver_use_df = pd.DataFrame(loading_receiver_use, index=["MI" + str(i+1) for i in range(loading_receiver_use.shape[0])],
                                        columns=processed.adata_list[0].var_names)
loading_sender_use_df = pd.DataFrame(loading_sender_use, index=["MI" + str(i+1) for i in range(loading_sender_use.shape[0])],
                                        columns=processed.adata_list[0].var_names)
loading_receiver_use_df_colsum = loading_receiver_use_df.abs().sum(axis=0)
loading_sender_use_df_colsum = loading_sender_use_df.abs().sum(axis=0)
loading_receiver_use_df_norm = loading_receiver_use_df.div(loading_receiver_use_df_colsum, axis=1)
loading_sender_use_df_norm = loading_sender_use_df.div(loading_sender_use_df_colsum, axis=1)


In [ ]:
loading_receiver_use_df_norm_choose = loading_receiver_use_df_norm.loc[MI_OI]
loading_sender_use_df_norm_choose = loading_sender_use_df_norm.loc[MI_OI]
loading_receiver_use_df_norm_choose = loading_receiver_use_df_norm_choose.sort_values(ascending=False)
loading_sender_use_df_norm_choose = loading_sender_use_df_norm_choose.sort_values(ascending=False)

In [ ]:
##Get the top regulators and targets based on the loading
top_targetgene_MIOI = loading_receiver_use_df_norm_choose.index[loading_receiver_use_df_norm_choose > 0.4].tolist()
print(top_targetgene_MIOI)
top_regulatorgene_MIOI = loading_sender_use_df_norm_choose.index[loading_sender_use_df_norm_choose > 0.4].tolist()
print(top_regulatorgene_MIOI)

### Top regulators

In [ ]:
go_res = gp.enrichr(
    gene_list=top_regulatorgene_MIOI,
    gene_sets=['GO_Biological_Process_2021'],
    organism="human",
    outdir="GO_MIOI_target",
    cutoff=0.05 
)

df_filter = go_res.results[
    (go_res.results['Adjusted P-value'] < 0.05)]

print(df_filter)

In [ ]:
df_filter["Term"].tolist()

In [ ]:
GO_term_choose = [
    # [1] Inflammatory / cytokine output axis (Sender)
    "positive regulation of interleukin-6 production (GO:0032755)",
    "positive regulation of interleukin-8 production (GO:0032757)",
    "positive regulation of chemokine production (GO:0032722)",

    # [2] Growth signal / kinase activation axis (Sender)
    "regulation of ERBB signaling pathway (GO:1901184)",
    "positive regulation of protein kinase B signaling (GO:0051897)",
    "positive regulation of MAPK cascade (GO:0043410)",

    # # [3] VEGF-remodeling coupling
    # "positive regulation of vascular endothelial growth factor production (GO:0010575)",
]

df_filter = df_filter[df_filter['Term'].isin(GO_term_choose)]
print(df_filter)
df_filter.to_csv(str(run_dirs['run_dir']) + '/GO_enrichment_topregulator_MI2.csv')

### Top targets

In [ ]:
go_res = gp.enrichr(
    gene_list=top_targetgene_MIOI,
    gene_sets=['GO_Biological_Process_2021'],
    organism="human",
    outdir="GO_MIOI_target", 
    cutoff=0.05
)

df_filter = go_res.results[
    (go_res.results['Adjusted P-value'] < 0.05)]

print(df_filter)

In [ ]:
df_filter["Term"].tolist()

In [ ]:
GO_term_choose = [
    # [1-1] Inflammatory / STAT response program (Receiver)
    "cellular response to cytokine stimulus (GO:0071345)",
    "regulation of receptor signaling pathway via STAT (GO:1904892)",

    # [1-2] Inflammation-driven remodeling program (Receiver)
    "extracellular matrix organization (GO:0030198)",
    
    # [2] Cell-cycle / proliferative execution program (Receiver)
    "DNA replication (GO:0006260)",
    "G2/M transition of mitotic cell cycle (GO:0000086)",
    
    # # [3] VEGF-remodeling coupling
    # "positive regulation of vascular endothelial growth factor receptor signaling pathway (GO:0030949)", 
]
df_filter = df_filter[df_filter['Term'].isin(GO_term_choose)]
print(df_filter)
df_filter.to_csv(str(run_dirs['run_dir']) + '/GO_enrichment_toptarget_MI2.csv')

In [ ]:
df_filter

## In-silico knockout of top regulators and targets based on the loading

In [ ]:
from pathlib import Path
import gc
import os
import pickle

import numpy as np
import pandas as pd
import torch

from SpiderNet.api import build_model

MI_PERTURB = "MI2"
NUM_TOP_FEATURES = 20
LOADING_THRESHOLD = 0.4
PERTURBATION_OUTPUT_DIR = run_dirs["run_dir"] / "MI2_spatial_perturbation"
PERTURBATION_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def ensure_processed_loaded():
    """Reload processed objects from disk if they are not already in memory."""
    if "processed" not in globals() or processed is None:
        from SpiderNet.io import load_processed_data
        return load_processed_data(run_dirs["run_dir"])
    return processed


processed = ensure_processed_loaded()


def ensure_model_loaded(processed_obj, checkpoint_path, cfg, device_name):
    """Build the model and load the trained checkpoint if the model is absent."""
    if "model" in globals() and model is not None:
        return model.to(device_name).eval()

    model_obj = build_model(
        processed=processed_obj,
        train_cfg=cfg,
        device=device_name,
    )

    checkpoint = torch.load(checkpoint_path, map_location=device_name)
    if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
        state_dict = checkpoint["model_state_dict"]
    elif isinstance(checkpoint, dict) and "state_dict" in checkpoint:
        state_dict = checkpoint["state_dict"]
    elif isinstance(checkpoint, dict):
        state_dict = checkpoint
    else:
        raise ValueError(
            "Unsupported checkpoint format. Expected a state-dict-like object, "
            f"but got: {type(checkpoint)}"
        )

    model_obj.load_state_dict(state_dict, strict=True)
    model_obj = model_obj.to(device_name)
    model_obj.eval()
    return model_obj


model = ensure_model_loaded(
    processed_obj=processed,
    checkpoint_path=reference_model_path,
    cfg=train_cfg,
    device_name=device,
)


def get_training_gene_names(processed_obj):
    """Return the training gene names as a NumPy string array."""
    if hasattr(processed_obj, "genenames_train") and processed_obj.genenames_train is not None:
        gene_names = processed_obj.genenames_train
        if hasattr(gene_names, "values"):
            gene_names = gene_names.values
        return np.asarray(gene_names).astype(str)
    return np.asarray(processed_obj.adata_list[0].var_names).astype(str)


GENE_NAMES_TRAIN = get_training_gene_names(processed)
GENE_INDEX_MAP = {gene: idx for idx, gene in enumerate(GENE_NAMES_TRAIN)}


def load_loading_matrices(run_dir, gene_names_train):
    """Load sender, receiver, and LR loading matrices from the saved result directory."""
    loading_receiver = np.load(run_dir / "loading_receiver_use.npy")
    loading_sender = np.load(run_dir / "loading_sender_use.npy")
    loading_lr = np.load(run_dir / "loading_LR_use.npy")

    mi_names = [f"MI{i + 1}" for i in range(loading_sender.shape[0])]

    loading_receiver_df = pd.DataFrame(loading_receiver, index=mi_names, columns=gene_names_train)
    loading_sender_df = pd.DataFrame(loading_sender, index=mi_names, columns=gene_names_train)

    with open(run_dir / "LR_list.pkl", "rb") as handle:
        lr_list = pickle.load(handle)

    lr_labels = []
    for ligand, receptor in lr_list:
        ligand_str = "+".join(np.asarray(ligand).astype(str).tolist())
        receptor_str = "+".join(np.asarray(receptor).astype(str).tolist())
        lr_labels.append(f"{ligand_str} -> {receptor_str}")

    loading_lr_df = pd.DataFrame(loading_lr, index=mi_names, columns=lr_labels)
    return loading_sender_df, loading_receiver_df, loading_lr_df


loading_sender_df, loading_receiver_df, loading_lr_df = load_loading_matrices(
    run_dirs["run_dir"],
    GENE_NAMES_TRAIN,
)


def normalize_by_column_sum(df, use_absolute=True):
    """Column-wise normalization with zero-safe denominators."""
    denom = df.abs().sum(axis=0) if use_absolute else df.sum(axis=0)
    denom = denom.replace(0, 1.0)
    return df.div(denom, axis=1).fillna(0.0)


loading_sender_norm = normalize_by_column_sum(loading_sender_df, use_absolute=True)
loading_receiver_norm = normalize_by_column_sum(loading_receiver_df, use_absolute=True)
loading_lr_norm = normalize_by_column_sum(loading_lr_df, use_absolute=False)


def get_top_feature_table(mi_name, n_top=20, loading_threshold=0.3):
    """Assemble the top ligands, receptors, sender genes, and receiver genes for one MI."""
    sender_series = loading_sender_norm.loc[mi_name].sort_values(ascending=False)
    receiver_series = loading_receiver_norm.loc[mi_name].sort_values(ascending=False)
    lr_series = loading_lr_norm.loc[mi_name].sort_values(ascending=False)

    top_sender = sender_series[sender_series > loading_threshold].head(n_top).index.tolist()
    top_receiver = receiver_series[receiver_series > loading_threshold].head(n_top).index.tolist()
    top_lr = lr_series[lr_series > loading_threshold].head(n_top).index.tolist()

    top_ligands = []
    top_receptors = []
    for lr_label in top_lr:
        ligand_str, receptor_str = lr_label.split(" -> ")
        top_ligands.extend([g for g in ligand_str.split("+") if g])
        top_receptors.extend([g for g in receptor_str.split("+") if g])

    top_ligands = sorted(pd.unique(pd.Series(top_ligands, dtype=str)).tolist()) if len(top_ligands) > 0 else []
    top_receptors = sorted(pd.unique(pd.Series(top_receptors, dtype=str)).tolist()) if len(top_receptors) > 0 else []

    topgene_df = pd.DataFrame(
        {
            "Genes": top_ligands + top_receptors + top_sender + top_receiver,
            "Type": (
                ["Ligand"] * len(top_ligands)
                + ["Receptor"] * len(top_receptors)
                + ["Top Sender"] * len(top_sender)
                + ["Top Receiver"] * len(top_receiver)
            ),
            "MI": mi_name,
        }
    ).drop_duplicates(ignore_index=True)

    return topgene_df, top_lr


topgene_df_mi2, toplr_mi2 = get_top_feature_table(
    MI_PERTURB,
    n_top=NUM_TOP_FEATURES,
    loading_threshold=LOADING_THRESHOLD,
)


def go_result_to_gene_table(go_result_df):
    """Expand a GO enrichment result table into a GO-term to gene mapping table."""
    gene_table = (
        go_result_df.loc[:, ["Term", "Genes"]]
        .assign(
            Gene=lambda d: (
                d["Genes"]
                .fillna("")
                .astype(str)
                .str.split(r"\s*;\s*")
            )
        )
        .explode("Gene")
        .rename(columns={"Term": "GO_Term"})
        .drop(columns=["Genes"])
    )
    gene_table["Gene"] = gene_table["Gene"].astype(str).str.strip()
    gene_table = gene_table[gene_table["Gene"] != ""].drop_duplicates(["Gene", "GO_Term"])
    return gene_table.reset_index(drop=True)



def keep_unique_genesets(go_gene_df):
    """Keep only one GO term for duplicated gene sets."""
    go_to_genes = go_gene_df.groupby("GO_Term")["Gene"].apply(
        lambda s: frozenset(str(x).strip() for x in s.dropna().unique())
    )

    seen_gene_sets = {}
    go_keep = []
    for go_term, gene_set in go_to_genes.items():
        if gene_set not in seen_gene_sets:
            seen_gene_sets[gene_set] = go_term
            go_keep.append(go_term)

    return go_gene_df[go_gene_df["GO_Term"].isin(go_keep)].copy().reset_index(drop=True)


sender_go_path = run_dirs["run_dir"] / "GO_enrichment_topregulator_MI2.csv"
receiver_go_path = run_dirs["run_dir"] / "GO_enrichment_toptarget_MI2.csv"

if not sender_go_path.exists() or not receiver_go_path.exists():
    raise FileNotFoundError(
        "GO enrichment tables for MI2 were not found. "
        "Please run the GO enrichment cells above before executing the perturbation section."
    )

sender_go_gene_table = keep_unique_genesets(
    go_result_to_gene_table(pd.read_csv(sender_go_path))
)
receiver_go_gene_table = keep_unique_genesets(
    go_result_to_gene_table(pd.read_csv(receiver_go_path))
)

excluded_genes = set(sender_go_gene_table["Gene"].tolist()) | set(receiver_go_gene_table["Gene"].tolist())
topgene_df_mi2_filtered = topgene_df_mi2[~topgene_df_mi2["Genes"].isin(excluded_genes)].copy().reset_index(drop=True)

knockout_upstream_genes = topgene_df_mi2_filtered[
    topgene_df_mi2_filtered["Type"].isin(["Ligand", "Top Sender"])
]["Genes"].tolist()
knockout_downstream_genes = topgene_df_mi2_filtered[
    topgene_df_mi2_filtered["Type"].isin(["Receptor", "Top Receiver"])
]["Genes"].tolist()

knockout_upstream_geneindex = [GENE_INDEX_MAP[g] for g in knockout_upstream_genes if g in GENE_INDEX_MAP]
knockout_downstream_geneindex = [GENE_INDEX_MAP[g] for g in knockout_downstream_genes if g in GENE_INDEX_MAP]

all_cell_types = sorted(
    pd.unique(
        pd.concat(
            [
                pd.Series(adata_i.obs[CELL_TYPE_COL].astype(str).values)
                for adata_i in processed.adata_list
            ],
            ignore_index=True,
        )
    ).tolist()
)
tumor_cell_types = sorted([ct for ct in all_cell_types if ct.endswith("-cancercell")])

batch_sample_ids = [
    str(np.asarray(processed.spidernet_data[i]["sample"])[0])
    for i in range(len(processed.spidernet_data))
]
batch_cancer_types = [sample_id.split("_")[0] for sample_id in batch_sample_ids]
unique_cancer_types = sorted(pd.unique(pd.Series(batch_cancer_types, dtype=str)).tolist())

print(f"Perturbation target: {MI_PERTURB}")
print(f"Number of upstream knockout genes: {len(knockout_upstream_geneindex)}")
print(f"Number of downstream knockout genes: {len(knockout_downstream_geneindex)}")
print("Tumor cell types used for both sender and receiver:")
print(tumor_cell_types)
print("Top MI2 feature table after removing GO genes:")
display(topgene_df_mi2_filtered)

In [ ]:
toplr_mi2

In [ ]:
from collections import OrderedDict


def to_device_copy(data_obj, device_name):
    """Clone a PyG data object and move it to the requested device."""
    data_copy = data_obj.clone()
    if hasattr(data_copy, "to"):
        data_copy = data_copy.to(device_name)
    return data_copy



def extract_reconstruction_tensor(model_output):
    """Extract the reconstructed expression tensor from different model-output formats."""
    if torch.is_tensor(model_output):
        return model_output

    if isinstance(model_output, (tuple, list)) and len(model_output) > 0:
        first_item = model_output[0]
        if torch.is_tensor(first_item):
            return first_item

    if isinstance(model_output, dict):
        for key in [
            "exprecon",
            "expression_reconstruction",
            "x_recon",
            "reconstruction",
            "expr_recon",
        ]:
            if key in model_output and torch.is_tensor(model_output[key]):
                return model_output[key]

    raise ValueError(
        "Could not identify the reconstructed expression tensor from the model output."
    )



def get_go_average_expression(expr_matrix, go_gene_table, gene_index_map):
    """Compute per-cell average reconstructed expression for each GO program."""
    go_avg = OrderedDict()
    for go_term in pd.unique(go_gene_table["GO_Term"]):
        genes_cur = go_gene_table.loc[go_gene_table["GO_Term"] == go_term, "Gene"].astype(str).tolist()
        gene_idx_cur = [gene_index_map[g] for g in genes_cur if g in gene_index_map]
        if len(gene_idx_cur) == 0:
            continue
        go_avg[go_term] = expr_matrix[:, gene_idx_cur].mean(axis=1)
    return go_avg



def append_nested_results(store_dict, outer_key, inner_dict):
    """Append arrays from one batch into a nested cancer-type dictionary."""
    if outer_key not in store_dict:
        store_dict[outer_key] = {}

    for inner_key, values in inner_dict.items():
        if inner_key not in store_dict[outer_key]:
            store_dict[outer_key][inner_key] = []
        store_dict[outer_key][inner_key].extend(np.asarray(values, dtype=float).tolist())


exprecon_diff_list_sender_cancertype = {}
exprecon_diff_list_receiver_cancertype = {}
perturbation_batch_summary = []

for cancer_type_cur in unique_cancer_types:
    batch_indices_cur = [
        i for i, ct in enumerate(batch_cancer_types)
        if ct == cancer_type_cur
    ]

    print(f"Running MI2 perturbation for cancer type: {cancer_type_cur} ({len(batch_indices_cur)} batches)")

    for batch_idx in batch_indices_cur:
        adata_batch = processed.adata_list[batch_idx]
        cell_types_batch = adata_batch.obs[CELL_TYPE_COL].astype(str).values

        sender_idx = np.where(np.isin(cell_types_batch, tumor_cell_types))[0]
        receiver_idx = np.where(np.isin(cell_types_batch, tumor_cell_types))[0]

        if sender_idx.size == 0 or receiver_idx.size == 0:
            perturbation_batch_summary.append(
                {
                    "CancerType": cancer_type_cur,
                    "BatchIndex": batch_idx,
                    "SampleID": batch_sample_ids[batch_idx],
                    "n_sender_cells": int(sender_idx.size),
                    "n_receiver_cells": int(receiver_idx.size),
                    "status": "skipped_no_tumor_cells",
                }
            )
            continue

        data_original = to_device_copy(processed.spidernet_data[batch_idx], device)
        data_knockout = to_device_copy(processed.spidernet_data[batch_idx], device)

        if len(knockout_upstream_geneindex) > 0:
            data_knockout.x[
                torch.as_tensor(sender_idx, device=data_knockout.x.device).unsqueeze(1),
                torch.as_tensor(knockout_upstream_geneindex, device=data_knockout.x.device),
            ] = 0.0

        if len(knockout_downstream_geneindex) > 0:
            data_knockout.x[
                torch.as_tensor(receiver_idx, device=data_knockout.x.device).unsqueeze(1),
                torch.as_tensor(knockout_downstream_geneindex, device=data_knockout.x.device),
            ] = 0.0

        with torch.no_grad():
            exprecon_original = extract_reconstruction_tensor(model(data_original))
            exprecon_knockout = extract_reconstruction_tensor(model(data_knockout))

        exprecon_original = exprecon_original.detach().cpu().numpy().astype(np.float32)
        exprecon_knockout = exprecon_knockout.detach().cpu().numpy().astype(np.float32)

        sender_original = exprecon_original[sender_idx, :]
        sender_knockout = exprecon_knockout[sender_idx, :]
        receiver_original = exprecon_original[receiver_idx, :]
        receiver_knockout = exprecon_knockout[receiver_idx, :]

        sender_go_original = get_go_average_expression(
            sender_original,
            sender_go_gene_table,
            GENE_INDEX_MAP,
        )
        sender_go_knockout = get_go_average_expression(
            sender_knockout,
            sender_go_gene_table,
            GENE_INDEX_MAP,
        )
        receiver_go_original = get_go_average_expression(
            receiver_original,
            receiver_go_gene_table,
            GENE_INDEX_MAP,
        )
        receiver_go_knockout = get_go_average_expression(
            receiver_knockout,
            receiver_go_gene_table,
            GENE_INDEX_MAP,
        )

        sender_go_diff = {
            go_term: sender_go_knockout[go_term] - sender_go_original[go_term]
            for go_term in sender_go_original.keys()
            if go_term in sender_go_knockout
        }
        receiver_go_diff = {
            go_term: receiver_go_knockout[go_term] - receiver_go_original[go_term]
            for go_term in receiver_go_original.keys()
            if go_term in receiver_go_knockout
        }

        append_nested_results(exprecon_diff_list_sender_cancertype, cancer_type_cur, sender_go_diff)
        append_nested_results(exprecon_diff_list_receiver_cancertype, cancer_type_cur, receiver_go_diff)

        perturbation_batch_summary.append(
            {
                "CancerType": cancer_type_cur,
                "BatchIndex": batch_idx,
                "SampleID": batch_sample_ids[batch_idx],
                "n_sender_cells": int(sender_idx.size),
                "n_receiver_cells": int(receiver_idx.size),
                "status": "finished",
            }
        )

        release_memory(
            "adata_batch",
            "cell_types_batch",
            "data_original",
            "data_knockout",
            "exprecon_original",
            "exprecon_knockout",
            "sender_original",
            "sender_knockout",
            "receiver_original",
            "receiver_knockout",
            "sender_go_original",
            "sender_go_knockout",
            "receiver_go_original",
            "receiver_go_knockout",
            "sender_go_diff",
            "receiver_go_diff",
            namespace=globals(),
            run_gc=True,
            clear_cuda=True,
        )

perturbation_batch_summary_df = pd.DataFrame(perturbation_batch_summary)
perturbation_batch_summary_df.to_csv(
    PERTURBATION_OUTPUT_DIR / f"{MI_PERTURB}_perturbation_batch_summary.csv",
    index=False,
)

with open(PERTURBATION_OUTPUT_DIR / f"{MI_PERTURB}_sender_go_diff.pkl", "wb") as handle:
    pickle.dump(exprecon_diff_list_sender_cancertype, handle)
with open(PERTURBATION_OUTPUT_DIR / f"{MI_PERTURB}_receiver_go_diff.pkl", "wb") as handle:
    pickle.dump(exprecon_diff_list_receiver_cancertype, handle)

print("Finished MI2 perturbation analysis.")
display(perturbation_batch_summary_df)

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import wilcoxon

mpl.rcParams.update(
    {
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "axes.linewidth": 0.8,
        "xtick.major.width": 0.8,
        "ytick.major.width": 0.8,
        "xtick.major.size": 3.5,
        "ytick.major.size": 3.5,
    }
)
sns.set_theme(style="white", context="paper")


def nested_dict_to_long_df(d, cancer_key="CancerType", go_key="GO", value_key="Value"):
    """Convert a nested cancer-type dictionary into a long-form DataFrame."""
    rows = []
    for cancer_type, go_dict in d.items():
        if go_dict is None:
            continue
        for go_term, vals in go_dict.items():
            if vals is None:
                continue
            if isinstance(vals, (list, tuple, np.ndarray, pd.Series)):
                for value in vals:
                    if pd.isna(value):
                        continue
                    rows.append(
                        {
                            cancer_key: cancer_type,
                            go_key: go_term,
                            value_key: float(value),
                        }
                    )
            else:
                if not pd.isna(vals):
                    rows.append(
                        {
                            cancer_key: cancer_type,
                            go_key: go_term,
                            value_key: float(vals),
                        }
                    )
    return pd.DataFrame(rows)


def wilcoxon_against_zero(values):
    """
    One-sample Wilcoxon signed-rank test against zero.

    Returns:
        dict with:
            - n
            - mean
            - median
            - p_less_0
            - p_greater_0
            - p_two_sided
    """
    values = np.asarray(values, dtype=float)
    values = values[~np.isnan(values)]
    n_vals = int(values.size)

    result_dict = {
        "n": n_vals,
        "mean": np.nan,
        "median": np.nan,
        "p_less_0": np.nan,
        "p_greater_0": np.nan,
        "p_two_sided": np.nan,
    }

    if n_vals == 0:
        return result_dict

    result_dict["mean"] = float(np.mean(values))
    result_dict["median"] = float(np.median(values))

    if n_vals < 3:
        return result_dict

    if np.allclose(values, 0):
        result_dict["p_less_0"] = 1.0
        result_dict["p_greater_0"] = 1.0
        result_dict["p_two_sided"] = 1.0
        return result_dict

    try:
        result_dict["p_less_0"] = float(
            wilcoxon(values, alternative="less", zero_method="wilcox").pvalue
        )
    except Exception:
        pass

    try:
        result_dict["p_greater_0"] = float(
            wilcoxon(values, alternative="greater", zero_method="wilcox").pvalue
        )
    except Exception:
        pass

    try:
        result_dict["p_two_sided"] = float(
            wilcoxon(values, alternative="two-sided", zero_method="wilcox").pvalue
        )
    except Exception:
        pass

    return result_dict


def wrap_go_title(go_term):
    """Move the GO accession to a second line for compact facet titles."""
    return re.sub(r"\s*(\(\s*GO:\d+\s*\))\s*$", r"\n\1", str(go_term).strip())


def compute_common_cancer_order(sender_df, receiver_df, value_key="Value"):
    """
    Compute a common cancer-type order using pooled sender + receiver values.

    Cancer types with larger decreases (more negative pooled mean changes)
    are placed to the left.
    """
    sender_tmp = sender_df.copy()
    sender_tmp["Side"] = "Sender"

    receiver_tmp = receiver_df.copy()
    receiver_tmp["Side"] = "Receiver"

    combined_df = pd.concat([sender_tmp, receiver_tmp], axis=0, ignore_index=True)

    overall_df = (
        combined_df.groupby("CancerType", observed=True)[value_key]
        .agg(["mean", "median", "count"])
        .reset_index()
        .rename(
            columns={
                "mean": "global_mean_change",
                "median": "global_median_change",
                "count": "global_n_values",
            }
        )
    )

    side_summary_df = (
        combined_df.groupby(["CancerType", "Side"], observed=True)[value_key]
        .agg(["mean", "median", "count"])
        .reset_index()
        .pivot(index="CancerType", columns="Side")
    )

    side_summary_df.columns = [
        f"{side.lower()}_{stat}"
        for stat, side in side_summary_df.columns.to_flat_index()
    ]
    side_summary_df = side_summary_df.reset_index()

    order_df = overall_df.merge(side_summary_df, on="CancerType", how="left")

    order_df = order_df.sort_values(
        ["global_mean_change", "global_median_change", "CancerType"],
        ascending=[True, True, True],
    ).reset_index(drop=True)

    order_df["common_order_rank"] = np.arange(1, len(order_df) + 1)
    cancer_order = order_df["CancerType"].tolist()

    return cancer_order, order_df


def build_wilcoxon_stats_df(df_long, side_name):
    """
    Run one-sample Wilcoxon signed-rank test against 0 for each GO x CancerType boxplot.
    """
    test_rows = []
    for (go_term, cancer_type), sub_df in df_long.groupby(["GO", "CancerType"], observed=True):
        stats_dict = wilcoxon_against_zero(sub_df["Value"].values)
        test_rows.append(
            {
                "Side": side_name,
                "GO": go_term,
                "CancerType": cancer_type,
                "n": stats_dict["n"],
                "mean": stats_dict["mean"],
                "median": stats_dict["median"],
                "p_less_0": stats_dict["p_less_0"],
                "p_greater_0": stats_dict["p_greater_0"],
                "p_two_sided": stats_dict["p_two_sided"],
                "is_sig_less_p05": (
                    stats_dict["p_less_0"] < 0.05 if pd.notna(stats_dict["p_less_0"]) else False
                ),
                "is_sig_two_sided_p05": (
                    stats_dict["p_two_sided"] < 0.05 if pd.notna(stats_dict["p_two_sided"]) else False
                ),
            }
        )

    test_df = pd.DataFrame(test_rows).sort_values(
        ["Side", "GO", "CancerType"]
    ).reset_index(drop=True)

    return test_df


def plot_knockout_go_boxplots(
    result_dict,
    output_prefix,
    fill_color,
    edge_color,
    cancer_order,
    side_name,
    facet_wrap=3,
):
    """
    Plot faceted boxplots of GO-program changes using a precomputed common cancer order.
    Also save the long-format data and per-boxplot Wilcoxon statistics.

    Parameters
    ----------
    edge_color : str
        Edge color for the boxplots.
    """
    df_long = nested_dict_to_long_df(result_dict)
    if df_long.empty:
        raise ValueError(f"No perturbation result is available for {output_prefix}.")

    df_long["Side"] = side_name
    df_long["CancerType"] = pd.Categorical(
        df_long["CancerType"],
        categories=cancer_order,
        ordered=True,
    )

    test_df = build_wilcoxon_stats_df(df_long, side_name=side_name)

    df_long.to_csv(str(output_prefix) + "_longformat.csv", index=False)
    test_df.to_csv(str(output_prefix) + "_wilcoxon_against0_results.csv", index=False)

    significance_lookup = {
        (row["GO"], row["CancerType"]): bool(row["is_sig_less_p05"])
        for _, row in test_df.iterrows()
    }

    grid = sns.FacetGrid(
        df_long,
        col="GO",
        col_wrap=facet_wrap,
        sharey=False,
        height=3.0,
        aspect=1.0,
        despine=False,
    )

    def facet_boxplot(data, **kwargs):
        ax = plt.gca()
        go_term = data["GO"].iloc[0]

        palette = {
            ct: (fill_color if significance_lookup.get((go_term, ct), False) else "#ECECEC")
            for ct in cancer_order
        }

        sns.boxplot(
            data=data,
            x="CancerType",
            y="Value",
            order=cancer_order,
            palette=palette,
            linewidth=0.8,
            showfliers=False,
            width=0.68,  # ~85% of seaborn default width 0.8
            ax=ax,
            boxprops=dict(edgecolor=edge_color, linewidth=0.8),
            whiskerprops=dict(color=edge_color, linewidth=0.8),
            capprops=dict(color=edge_color, linewidth=0.0, alpha=0.0),  # hide top/bottom caps
            medianprops=dict(color=edge_color, linewidth=0.8),
        )

        ax.axhline(y=0, linestyle="--", linewidth=0.9, color="0.6", zorder=0)

        ax.tick_params(axis="x", pad=2)
        plt.setp(
            ax.get_xticklabels(),
            rotation=45,
            ha="right",
            va="top",
            rotation_mode="anchor",
        )

        sns.despine(ax=ax)

    grid.map_dataframe(facet_boxplot)

    for ax in grid.axes.flatten():
        raw_title = ax.get_title()
        raw_term = raw_title.split("=", 1)[1].strip() if "=" in raw_title else raw_title.strip()
        ax.set_title("GO = " + wrap_go_title(raw_term))

    grid.set_axis_labels("Cancer type", "Knockout - original")
    plt.tight_layout()
    plt.savefig(str(output_prefix) + ".pdf", bbox_inches="tight")
    plt.savefig(str(output_prefix) + ".png", dpi=300, bbox_inches="tight")
    plt.show()

    return df_long, test_df


# ============================================================
# Build sender / receiver long-format data first
# ============================================================
sender_long_for_order = nested_dict_to_long_df(exprecon_diff_list_sender_cancertype)
receiver_long_for_order = nested_dict_to_long_df(exprecon_diff_list_receiver_cancertype)

if sender_long_for_order.empty:
    raise ValueError("Sender perturbation result is empty.")
if receiver_long_for_order.empty:
    raise ValueError("Receiver perturbation result is empty.")


# ============================================================
# Compute one common cancer-type order from pooled sender + receiver values
# ============================================================
common_cancer_order, common_cancer_order_df = compute_common_cancer_order(
    sender_df=sender_long_for_order,
    receiver_df=receiver_long_for_order,
    value_key="Value",
)

common_cancer_order_df.to_csv(
    PERTURBATION_OUTPUT_DIR / f"{MI_PERTURB}_common_cancer_order_summary.csv",
    index=False,
)


# ============================================================
# Plot sender with the common order
# ============================================================
sender_long_df, sender_test_df = plot_knockout_go_boxplots(
    result_dict=exprecon_diff_list_sender_cancertype,
    output_prefix=PERTURBATION_OUTPUT_DIR / f"{MI_PERTURB}_sender_GO_boxplot",
    fill_color="#b7e3f9",
    edge_color="#9BBBE2",
    cancer_order=common_cancer_order,
    side_name="Sender",
    facet_wrap=3,
)


# ============================================================
# Plot receiver with the common order
# ============================================================
receiver_long_df, receiver_test_df = plot_knockout_go_boxplots(
    result_dict=exprecon_diff_list_receiver_cancertype,
    output_prefix=PERTURBATION_OUTPUT_DIR / f"{MI_PERTURB}_receiver_GO_boxplot",
    fill_color="#f6d7e2",
    edge_color="#D8B5D6",
    cancer_order=common_cancer_order,
    side_name="Receiver",
    facet_wrap=3,
)


# ============================================================
# Combine sender + receiver Wilcoxon results into one dataframe
# ============================================================
combined_wilcoxon_df = pd.concat(
    [sender_test_df, receiver_test_df],
    axis=0,
    ignore_index=True,
).sort_values(["Side", "GO", "CancerType"]).reset_index(drop=True)

combined_wilcoxon_df.to_csv(
    PERTURBATION_OUTPUT_DIR / f"{MI_PERTURB}_sender_receiver_combined_wilcoxon_against0_results.csv",
    index=False,
)


# ============================================================
# Optional: combine long-format data as well
# ============================================================
combined_long_df = pd.concat(
    [sender_long_df, receiver_long_df],
    axis=0,
    ignore_index=True,
).sort_values(["Side", "GO", "CancerType"]).reset_index(drop=True)

combined_long_df.to_csv(
    PERTURBATION_OUTPUT_DIR / f"{MI_PERTURB}_sender_receiver_combined_longformat.csv",
    index=False,
)


# ============================================================
# Release memory
# ============================================================
release_memory(
    "loading_sender_df",
    "loading_receiver_df",
    "loading_lr_df",
    "loading_sender_norm",
    "loading_receiver_norm",
    "loading_lr_norm",
    namespace=globals(),
    run_gc=True,
    clear_cuda=False,
)

In [ ]:
# (combined_wilcoxon_df['p_less_0']).tolist()
combined_wilcoxon_df